In [ ]:
!pip install rasterio pyproj plotly -q

from google.colab import drive
drive.mount('/content/drive')

import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from rasterio.windows import Window
from rasterio.warp import transform_bounds
import plotly.graph_objects as go

In [ ]:
DEM_DIR = Path("/content/drive/MyDrive/EcoSignal_DEM")

dem_2015_path = DEM_DIR / "Dem_2015.tif"
dem_2022_path = DEM_DIR / "Dem_2022.tif"
dem_diff_path = DEM_DIR / "Dem_2013.tif"

for path in [dem_2015_path, dem_2022_path, dem_diff_path]:
    print("\nFILE:", path.name)
    with rasterio.open(path) as src:
        print("CRS:", src.crs)
        print("Width x Height:", src.width, "x", src.height)
        print("Resolution:", src.res)
        print("Bounds:", src.bounds)
        print("Nodata:", src.nodata)

In [ ]:
def read_center_crop(path, crop_size=1200):
    """
    Reads a square crop from the center of a huge DEM.
    crop_size=1200 means 1200 x 1200 pixels.
    """
    with rasterio.open(path) as src:
        center_x = src.width // 2
        center_y = src.height // 2

        half = crop_size // 2

        window = Window(
            center_x - half,
            center_y - half,
            crop_size,
            crop_size
        )

        data = src.read(1, window=window, masked=True)
        transform = src.window_transform(window)
        profile = src.profile.copy()

        profile.update({
            "height": crop_size,
            "width": crop_size,
            "transform": transform
        })

        return data, transform, profile

dem_2015, transform_2015, profile_2015 = read_center_crop(dem_2015_path, crop_size=1200)
dem_2022, transform_2022, profile_2022 = read_center_crop(dem_2022_path, crop_size=1200)
dem_diff, transform_diff, profile_diff = read_center_crop(dem_diff_path, crop_size=1200)

print("Cropped DEM shape:", dem_2022.shape)

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(dem_diff, cmap="RdBu", vmin=-2, vmax=2)
plt.colorbar(label="Elevation change, meters")
plt.title("Observed Elevation Change: 2015–2022")
plt.axis("off")
plt.show()

In [ ]:
def calculate_slope_degrees(dem, transform):
    """
    Calculates slope in degrees from a DEM crop.
    """
    dem_array = dem.filled(np.nan)

    xres = abs(transform.a)
    yres = abs(transform.e)

    dz_dy, dz_dx = np.gradient(dem_array, yres, xres)

    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    slope_deg = np.degrees(slope_rad)

    return slope_deg

slope_2022 = calculate_slope_degrees(dem_2022, transform_2022)

plt.figure(figsize=(10, 8))
plt.imshow(slope_2022, cmap="terrain")
plt.colorbar(label="Slope, degrees")
plt.title("Slope Map from 2022 DEM")
plt.axis("off")
plt.show()

In [ ]:
# Convert arrays to usable form
change = dem_diff.filled(np.nan)

# Slope risk: steeper areas are more vulnerable
slope_risk = np.clip(slope_2022 / 35, 0, 1)

# Past erosion risk: areas that already lost elevation are more vulnerable
past_erosion = np.where(change < 0, np.abs(change), 0)
past_erosion_risk = np.clip(past_erosion / 1.0, 0, 1)

# Simple MVP geomorphology risk
geomorph_risk = (0.55 * slope_risk) + (0.45 * past_erosion_risk)

plt.figure(figsize=(10, 8))
plt.imshow(geomorph_risk, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="Geomorphology risk score")
plt.title("MVP Geomorphology Risk Map")
plt.axis("off")
plt.show()

In [ ]:
years_between = 2022 - 2015
annual_change_rate = change / years_between

future_year = 2032
years_forward = future_year - 2022

future_dem_2032 = dem_2022.filled(np.nan) + (annual_change_rate * years_forward)

plt.figure(figsize=(10, 8))
plt.imshow(future_dem_2032, cmap="terrain")
plt.colorbar(label="Predicted elevation")
plt.title("Simulated Future DEM: 2032")
plt.axis("off")
plt.show()

In [ ]:
# Downsample for speed
step = 10

z_current = dem_2022.filled(np.nan)[::step, ::step]
z_future = future_dem_2032[::step, ::step]

rows, cols = z_current.shape
x = np.arange(cols)
y = np.arange(rows)

fig = go.Figure()

fig.add_trace(go.Surface(
    z=z_current,
    x=x,
    y=y,
    colorscale="Earth",
    name="Current Terrain",
    showscale=True
))


fig.update_layout(
    title="Current 3D Terrain Surface, 2022",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Elevation"
    ),
    width=900,
    height=700
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Surface(
    z=z_future,
    x=x,
    y=y,
    colorscale="Earth",
    name="Future Terrain",
    showscale=True
))

fig.update_layout(
    title="Predicted 3D Terrain Surface, 2032",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Predicted Elevation"
    ),
    width=900,
    height=700
)

fig.show()

V2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Convert masked arrays to normal arrays
d15 = dem_2015.filled(np.nan)
d22 = dem_2022.filled(np.nan)
dd_file = dem_diff.filled(np.nan)

# Calculate our own difference
dd_calc = d22 - d15

valid = np.isfinite(dd_calc) & np.isfinite(dd_file)

corr = np.corrcoef(dd_calc[valid].ravel(), dd_file[valid].ravel())[0, 1]
print("Correlation between calculated diff and file diff:", corr)

if corr < 0:
    print("DEM difference file appears reversed. Flipping sign.")
    change = -dd_file
else:
    print("DEM difference file matches 2022 - 2015.")
    change = dd_file

In [ ]:
lod = 0.15  # limit of detection in meters

clean_change = np.where(np.abs(change) < lod, 0, change)

plt.figure(figsize=(10, 8))
plt.imshow(clean_change, cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar(label="Cleaned elevation change, meters")
plt.title("Cleaned DEM Difference: Erosion and Deposition")
plt.axis("off")
plt.show()

In [ ]:
change_class = np.zeros_like(clean_change)

change_class[clean_change < -lod] = -1   # erosion
change_class[clean_change > lod] = 1     # deposition
change_class[clean_change == 0] = 0      # stable

plt.figure(figsize=(10, 8))
plt.imshow(change_class, cmap="bwr", vmin=-1, vmax=1)
plt.colorbar(label="-1 erosion, 0 stable, 1 deposition")
plt.title("Classified Terrain Change Zones")
plt.axis("off")
plt.show()

In [ ]:
def calculate_slope_degrees(dem_array, transform):
    xres = abs(transform.a)
    yres = abs(transform.e)

    dz_dy, dz_dx = np.gradient(dem_array, yres, xres)

    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    slope_deg = np.degrees(slope_rad)

    return slope_deg

slope_2022 = calculate_slope_degrees(d22, transform_2022)

plt.figure(figsize=(10, 8))
plt.imshow(slope_2022, cmap="terrain")
plt.colorbar(label="Slope, degrees")
plt.title("Slope Map from 2022 DEM")
plt.axis("off")
plt.show()

In [ ]:
def normalize_percentile(arr, low=5, high=95):
    valid = np.isfinite(arr)
    p_low, p_high = np.nanpercentile(arr[valid], [low, high])
    norm = (arr - p_low) / (p_high - p_low)
    return np.clip(norm, 0, 1)

# Risk from slope
slope_risk = normalize_percentile(slope_2022)

# Risk from past erosion
past_erosion = np.where(clean_change < 0, np.abs(clean_change), 0)
past_erosion_risk = normalize_percentile(past_erosion)

# MVP geomorphology risk
geomorph_risk = (0.55 * slope_risk) + (0.45 * past_erosion_risk)

plt.figure(figsize=(10, 8))
plt.imshow(geomorph_risk, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="Geomorphology risk score")
plt.title("MVP Geomorphology Risk Map")
plt.axis("off")
plt.show()

In [ ]:
years_between = 2022 - 2015

annual_change_rate = clean_change / years_between

future_year = 2032
years_forward = future_year - 2022

future_dem_2032 = d22 + (annual_change_rate * years_forward)

plt.figure(figsize=(10, 8))
plt.imshow(future_dem_2032, cmap="terrain")
plt.colorbar(label="Predicted elevation")
plt.title("Simulated Future DEM: 2032")
plt.axis("off")
plt.show()

In [ ]:
import plotly.graph_objects as go

step = 10  # increase to 20 if Colab is slow

z_current = d22[::step, ::step]

rows, cols = z_current.shape
x = np.arange(cols)
y = np.arange(rows)

fig = go.Figure()

fig.add_trace(go.Surface(
    z=z_current,
    x=x,
    y=y,
    colorscale="Earth",
    showscale=True
))

fig.update_layout(
    title="Current 3D Terrain Surface: 2022",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Elevation"
    ),
    width=900,
    height=700
)

fig.show()

In [ ]:
z_future = future_dem_2032[::step, ::step]

fig = go.Figure()

fig.add_trace(go.Surface(
    z=z_future,
    x=x,
    y=y,
    colorscale="Earth",
    showscale=True
))

fig.update_layout(
    title="Predicted 3D Terrain Surface: 2032",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Predicted elevation"
    ),
    width=900,
    height=700
)

fig.show()

In [ ]:
future_change_2022_2032 = future_dem_2032 - d22

plt.figure(figsize=(10, 8))
plt.imshow(future_change_2022_2032, cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar(label="Predicted elevation change, meters")
plt.title("Predicted Terrain Change: 2022–2032")
plt.axis("off")
plt.show()

W Lichen Factors 70/30 Split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from rasterio.warp import transform as transform_coords
from scipy.ndimage import gaussian_filter

lichen_csv = DEM_DIR / "lichen_points.csv"

lichen_df = pd.read_csv(lichen_csv)
lichen_df.head()

In [ ]:
def growth_form_stress(growth_form):
    """
    Placeholder ecological weighting for MVP.
    You can tune this later with real literature/species labels.
    Higher = more stressed/vulnerable signal.
    """
    gf = str(growth_form).lower().strip()

    if gf == "crustose":
        return 0.45
    elif gf == "foliose":
        return 0.25
    elif gf == "fruticose":
        return 0.15
    else:
        return 0.30

def calculate_lichen_stress(row):
    coverage = row["coverage_percent"]
    chlorosis = row["chlorosis_score"]
    nitrophyte = row["nitrophyte_score"]
    growth_stress = growth_form_stress(row["growth_form"])

    # Low coverage can suggest poor colonization or disturbance.
    coverage_stress = 1 - np.clip(coverage / 80, 0, 1)

    # Weighted MVP formula
    stress = (
        0.35 * chlorosis +
        0.25 * coverage_stress +
        0.25 * nitrophyte +
        0.15 * growth_stress
    )

    return float(np.clip(stress, 0, 1))

lichen_df["lichen_stress"] = lichen_df.apply(calculate_lichen_stress, axis=1)

lichen_df

In [ ]:
import rasterio

def lichen_points_to_pixels(df, raster_profile, raster_transform):
    """
    Converts lichen lat/lon points to row/col pixel positions
    in the DEM raster coordinate system.
    """
    raster_crs = raster_profile["crs"]

    lons = df["longitude"].values
    lats = df["latitude"].values

    # Convert WGS84 lon/lat to DEM CRS
    xs, ys = transform_coords("EPSG:4326", raster_crs, lons, lats)

    rows = []
    cols = []

    for x, y in zip(xs, ys):
        row, col = rasterio.transform.rowcol(raster_transform, x, y)
        rows.append(row)
        cols.append(col)

    df = df.copy()
    df["row"] = rows
    df["col"] = cols

    return df

lichen_pixels = lichen_points_to_pixels(
    lichen_df,
    profile_2022,
    transform_2022
)

lichen_pixels

In [ ]:
height, width = d22.shape

inside = (
    (lichen_pixels["row"] >= 0) &
    (lichen_pixels["row"] < height) &
    (lichen_pixels["col"] >= 0) &
    (lichen_pixels["col"] < width)
)

lichen_pixels["inside_crop"] = inside

print(lichen_pixels[["latitude", "longitude", "row", "col", "lichen_stress", "inside_crop"]])
print("Points inside crop:", inside.sum(), "out of", len(lichen_pixels))

In [ ]:
from scipy.ndimage import gaussian_filter
import numpy as np

def create_lichen_stress_surface(lichen_pixels, shape, sigma=50):
    """
    Turns lichen point stress values into a smooth raster surface.

    lichen_pixels must contain:
    - row
    - col
    - lichen_stress
    - cv_confidence
    - inside_crop

    shape should be the DEM shape, usually d22.shape.
    sigma controls how far each lichen point spreads across the map.
    """
    stress_grid = np.zeros(shape, dtype="float32")
    weight_grid = np.zeros(shape, dtype="float32")

    valid_points = lichen_pixels[lichen_pixels["inside_crop"] == True]

    for _, point in valid_points.iterrows():
        r = int(point["row"])
        c = int(point["col"])
        stress = float(point["lichen_stress"])
        confidence = float(point["cv_confidence"])

        stress_grid[r, c] += stress * confidence
        weight_grid[r, c] += confidence

    smooth_stress = gaussian_filter(stress_grid, sigma=sigma)
    smooth_weight = gaussian_filter(weight_grid, sigma=sigma)

    lichen_surface = np.divide(
        smooth_stress,
        smooth_weight,
        out=np.zeros_like(smooth_stress),
        where=smooth_weight > 0
    )

    return np.clip(lichen_surface, 0, 1)

In [ ]:
# Ensure test_lichen_pixels is defined before use
test_lichen_pixels = create_test_lichen_points(d22.shape)

lichen_stress_surface = create_lichen_stress_surface(
    test_lichen_pixels,
    d22.shape,
    sigma=60
)

Synthetic Lichen Data(not GPS Aligned)

In [ ]:
def create_test_lichen_points(shape):
    """
    Creates synthetic lichen stress points inside the current DEM crop.
    Use only for testing the pipeline.
    """
    h, w = shape

    test_points = pd.DataFrame({
        "row": [int(h*0.25), int(h*0.40), int(h*0.65), int(h*0.75)],
        "col": [int(w*0.30), int(w*0.55), int(w*0.45), int(w*0.70)],
        "lichen_stress": [0.85, 0.65, 0.25, 0.45],
        "cv_confidence": [0.92, 0.88, 0.91, 0.84],
        "inside_crop": [True, True, True, True]
    })

    return test_points

test_lichen_pixels = create_test_lichen_points(d22.shape)

lichen_stress_surface = create_lichen_stress_surface(
    test_lichen_pixels,
    d22.shape,
    sigma=60
)

plt.figure(figsize=(10, 8))
plt.imshow(lichen_stress_surface, cmap="YlOrRd", vmin=0, vmax=1)
plt.colorbar(label="Lichen stress score")
plt.title("Test Lichen Stress Surface")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(clean_change, cmap="RdBu", vmin=-1, vmax=1)
plt.imshow(lichen_stress_surface, cmap="YlOrRd", alpha=0.45, vmin=0, vmax=1)

plt.colorbar(label="Overlay: lichen stress on terrain change")
plt.title("Lichen Stress Over Observed Terrain Change")
plt.axis("off")
plt.show()

In [ ]:
ecosignal_risk = (
    0.70 * geomorph_risk +
    0.30 * lichen_stress_surface
)

ecosignal_risk = np.clip(ecosignal_risk, 0, 1)

plt.figure(figsize=(10, 8))
plt.imshow(ecosignal_risk, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="EcoSignal future land-change risk")
plt.title("EcoSignal Risk Map: Geomorphology + Lichen Stress")
plt.axis("off")
plt.show()

In [ ]:
years_between = 2022 - 2015
annual_change_rate = clean_change / years_between

future_year = 2032
years_forward = future_year - 2022

# Risk modifier:
# Low risk = slower projected change
# High risk = stronger projected change
risk_modifier = 0.5 + ecosignal_risk

lichen_informed_future_dem_2032 = d22 + (
    annual_change_rate * years_forward * risk_modifier
)

plt.figure(figsize=(10, 8))
plt.imshow(lichen_informed_future_dem_2032, cmap="terrain")
plt.colorbar(label="Predicted elevation")
plt.title("Lichen-Informed Simulated Future DEM: 2032")
plt.axis("off")
plt.show()

In [ ]:
difference_due_to_lichen = lichen_informed_future_dem_2032 - future_dem_2032

plt.figure(figsize=(10, 8))
plt.imshow(difference_due_to_lichen, cmap="PuOr", vmin=-0.5, vmax=0.5)
plt.colorbar(label="Elevation difference, meters")
plt.title("Impact of Lichen Stress on Future Terrain Prediction")
plt.axis("off")
plt.show()

In [ ]:
import plotly.graph_objects as go

step = 10

z_lichen_future = lichen_informed_future_dem_2032[::step, ::step]

rows, cols = z_lichen_future.shape
x = np.arange(cols)
y = np.arange(rows)

fig = go.Figure()

fig.add_trace(go.Surface(
    z=z_lichen_future,
    x=x,
    y=y,
    surfacecolor=ecosignal_risk[::step, ::step],
    colorscale="Inferno",
    showscale=True,
    colorbar=dict(title="EcoSignal risk")
))

fig.update_layout(
    title="EcoSignal 3D Future Landscape: 2032",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Predicted elevation"
    ),
    width=950,
    height=750
)

fig.show()

In [ ]:
points_for_plot = lichen_pixels[lichen_pixels["inside_crop"] == True].copy()

# If no real points are inside, fallback to test points
if len(points_for_plot) == 0:
    points_for_plot = test_lichen_pixels.copy()

point_rows = points_for_plot["row"].astype(int).values
point_cols = points_for_plot["col"].astype(int).values

# Convert full-resolution pixel positions to downsampled plot positions
plot_x = point_cols / step
plot_y = point_rows / step

# Get elevation at each point
plot_z = []
for r, c in zip(point_rows, point_cols):
    plot_z.append(lichen_informed_future_dem_2032[r, c] + 1.0)

plot_z = np.array(plot_z)

fig = go.Figure()

fig.add_trace(go.Surface(
    z=z_lichen_future,
    x=x,
    y=y,
    surfacecolor=ecosignal_risk[::step, ::step],
    colorscale="Inferno",
    showscale=True,
    colorbar=dict(title="EcoSignal risk")
))

fig.add_trace(go.Scatter3d(
    x=plot_x,
    y=plot_y,
    z=plot_z,
    mode="markers",
    marker=dict(
        size=6,
        color=points_for_plot["lichen_stress"],
        colorscale="YlOrRd",
        cmin=0,
        cmax=1,
        colorbar=dict(title="Lichen stress")
    ),
    text=[
        f"Lichen stress: {s:.2f}<br>Confidence: {c:.2f}"
        for s, c in zip(points_for_plot["lichen_stress"], points_for_plot["cv_confidence"])
    ],
    hoverinfo="text",
    name="Lichen observations"
))

fig.update_layout(
    title="EcoSignal 3D Future Landscape with Lichen Bioindicators",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Predicted elevation"
    ),
    width=1000,
    height=800
)

fig.show()

GPS Aligned Maps

In [ ]:
!pip install rasterio scipy plotly -q

import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from pathlib import Path
from rasterio.windows import from_bounds, Window
from rasterio.warp import transform as warp_transform
from rasterio.warp import transform_bounds
from scipy.ndimage import gaussian_filter

DEM_DIR = Path("/content/drive/MyDrive/EcoSignal_DEM")

dem_2015_path = DEM_DIR / "Dem_2015.tif"
dem_2022_path = DEM_DIR / "Dem_2022.tif"
dem_diff_path = DEM_DIR / "DemDif_2015-2022.tif"
lichen_csv = DEM_DIR / "lichen_points.csv"

lichen_df = pd.read_csv(lichen_csv)

print(lichen_df.head())

In [ ]:
def get_wgs84_bounds_from_lichen_points(df, buffer_m=300):
    """
    Creates a latitude/longitude bounding box around all lichen points.
    buffer_m adds extra space around the points.
    """
    min_lat = df["latitude"].min()
    max_lat = df["latitude"].max()
    min_lon = df["longitude"].min()
    max_lon = df["longitude"].max()

    mean_lat = np.deg2rad((min_lat + max_lat) / 2)

    lat_buffer = buffer_m / 111_320
    lon_buffer = buffer_m / (111_320 * np.cos(mean_lat))

    return (
        min_lon - lon_buffer,
        min_lat - lat_buffer,
        max_lon + lon_buffer,
        max_lat + lat_buffer
    )

crop_bounds_wgs84 = get_wgs84_bounds_from_lichen_points(
    lichen_df,
    buffer_m=500
)

print("Crop bounds in WGS84:")
print(crop_bounds_wgs84)

In [ ]:
def read_crop_by_wgs84_bounds(path, bounds_wgs84):
    """
    Reads a DEM crop using latitude/longitude bounds.
    Returns:
    - masked data array
    - crop transform
    - crop profile
    """
    with rasterio.open(path) as src:
        raster_bounds = transform_bounds(
            "EPSG:4326",
            src.crs,
            *bounds_wgs84,
            densify_pts=21
        )

        window = from_bounds(
            *raster_bounds,
            transform=src.transform
        )

        window = window.round_offsets().round_lengths()

        full_window = Window(0, 0, src.width, src.height)

        try:
            window = window.intersection(full_window)
        except Exception:
            raise ValueError(
                f"The requested lat/lon crop does not overlap {path.name}. "
                "Your lichen coordinates may be outside this DEM."
            )

        data = src.read(1, window=window, masked=True)
        transform = src.window_transform(window)

        profile = src.profile.copy()
        profile.update({
            "height": data.shape[0],
            "width": data.shape[1],
            "transform": transform
        })

        return data, transform, profile

dem_2015, transform_2015, profile_2015 = read_crop_by_wgs84_bounds(
    dem_2015_path,
    crop_bounds_wgs84
)

dem_2022, transform_2022, profile_2022 = read_crop_by_wgs84_bounds(
    dem_2022_path,
    crop_bounds_wgs84
)

dem_diff, transform_diff, profile_diff = read_crop_by_wgs84_bounds(
    dem_diff_path,
    crop_bounds_wgs84
)

print("2015 crop shape:", dem_2015.shape)
print("2022 crop shape:", dem_2022.shape)
print("Diff crop shape:", dem_diff.shape)
print("Crop CRS:", profile_2022["crs"])

In [ ]:
def lichen_points_to_pixels(df, raster_profile, raster_transform):
    """
    Converts lichen latitude/longitude coordinates into row/col pixels
    inside the cropped DEM.
    """
    raster_crs = raster_profile["crs"]

    lons = df["longitude"].values
    lats = df["latitude"].values

    xs, ys = warp_transform(
        "EPSG:4326",
        raster_crs,
        lons,
        lats
    )

    rows = []
    cols = []

    for x, y in zip(xs, ys):
        row, col = rasterio.transform.rowcol(
            raster_transform,
            x,
            y
        )
        rows.append(row)
        cols.append(col)

    out = df.copy()
    out["row"] = rows
    out["col"] = cols

    height = raster_profile["height"]
    width = raster_profile["width"]

    out["inside_crop"] = (
        (out["row"] >= 0) &
        (out["row"] < height) &
        (out["col"] >= 0) &
        (out["col"] < width)
    )

    return out

lichen_pixels = lichen_points_to_pixels(
    lichen_df,
    profile_2022,
    transform_2022
)

print(lichen_pixels[[
    "latitude",
    "longitude",
    "row",
    "col",
    "inside_crop"
]])

print("Points inside crop:", lichen_pixels["inside_crop"].sum(), "out of", len(lichen_pixels))

In [ ]:
d15 = dem_2015.filled(np.nan)
d22 = dem_2022.filled(np.nan)
dd_file = dem_diff.filled(np.nan)

# Confirm direction of DEM difference
dd_calc = d22 - d15

valid = np.isfinite(dd_calc) & np.isfinite(dd_file)

corr = np.corrcoef(
    dd_calc[valid].ravel(),
    dd_file[valid].ravel()
)[0, 1]

print("Correlation between calculated diff and file diff:", corr)

if corr < 0:
    print("DEM difference file appears reversed. Flipping sign.")
    change = -dd_file
else:
    print("DEM difference file matches 2022 - 2015.")
    change = dd_file

# Limit of detection
lod = 0.15

clean_change = np.where(
    np.abs(change) < lod,
    0,
    change
)

plt.figure(figsize=(10, 8))
plt.imshow(clean_change, cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar(label="Cleaned elevation change, meters")
plt.title("Aligned DEM Difference Crop")
plt.axis("off")
plt.show()

In [ ]:
def calculate_slope_degrees(dem_array, transform):
    xres = abs(transform.a)
    yres = abs(transform.e)

    dz_dy, dz_dx = np.gradient(dem_array, yres, xres)

    slope_rad = np.arctan(
        np.sqrt(dz_dx**2 + dz_dy**2)
    )

    slope_deg = np.degrees(slope_rad)

    return slope_deg

def normalize_percentile(arr, low=5, high=95):
    valid = np.isfinite(arr)

    p_low, p_high = np.nanpercentile(
        arr[valid],
        [low, high]
    )

    norm = (arr - p_low) / (p_high - p_low)

    return np.clip(norm, 0, 1)

slope_2022 = calculate_slope_degrees(
    d22,
    transform_2022
)

slope_risk = normalize_percentile(slope_2022)

past_erosion = np.where(
    clean_change < 0,
    np.abs(clean_change),
    0
)

past_erosion_risk = normalize_percentile(past_erosion)

geomorph_risk = (
    0.55 * slope_risk +
    0.45 * past_erosion_risk
)

geomorph_risk = np.clip(geomorph_risk, 0, 1)

plt.figure(figsize=(10, 8))
plt.imshow(geomorph_risk, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="Geomorphology risk")
plt.title("Aligned Geomorphology Risk Map")
plt.axis("off")
plt.show()

In [ ]:
def growth_form_stress(growth_form):
    gf = str(growth_form).lower().strip()

    if gf == "crustose":
        return 0.45
    elif gf == "foliose":
        return 0.25
    elif gf == "fruticose":
        return 0.15
    else:
        return 0.30

def calculate_lichen_stress(row):
    coverage = row["coverage_percent"]
    chlorosis = row["chlorosis_score"]
    nitrophyte = row["nitrophyte_score"]
    growth_stress = growth_form_stress(row["growth_form"])

    coverage_stress = 1 - np.clip(coverage / 80, 0, 1)

    stress = (
        0.35 * chlorosis +
        0.25 * coverage_stress +
        0.25 * nitrophyte +
        0.15 * growth_stress
    )

    return float(np.clip(stress, 0, 1))

lichen_pixels["lichen_stress"] = lichen_pixels.apply(
    calculate_lichen_stress,
    axis=1
)

lichen_pixels[[
    "latitude",
    "longitude",
    "coverage_percent",
    "chlorosis_score",
    "nitrophyte_score",
    "growth_form",
    "lichen_stress",
    "cv_confidence"
]]

Redefining geomorph_risk for the GPS-aligned crop

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Make sure these are from the GPS-adjusted crop
# d22 = dem_2022.filled(np.nan)
# clean_change = GPS-adjusted cleaned DEM difference
# transform_2022 = GPS-adjusted crop transform

def calculate_slope_degrees(dem_array, transform):
    """
    Calculates slope in degrees from the GPS-adjusted DEM crop.
    """
    xres = abs(transform.a)
    yres = abs(transform.e)

    dz_dy, dz_dx = np.gradient(dem_array, yres, xres)

    slope_rad = np.arctan(
        np.sqrt(dz_dx**2 + dz_dy**2)
    )

    slope_deg = np.degrees(slope_rad)

    return slope_deg


def normalize_percentile(arr, low=5, high=95):
    """
    Normalizes an array from 0 to 1 using percentile clipping.
    """
    valid = np.isfinite(arr)

    p_low, p_high = np.nanpercentile(
        arr[valid],
        [low, high]
    )

    if p_high == p_low:
        return np.zeros_like(arr, dtype="float32")

    norm = (arr - p_low) / (p_high - p_low)

    return np.clip(norm, 0, 1)


# 1. Recalculate slope from the GPS-adjusted 2022 crop
slope_2022 = calculate_slope_degrees(
    d22,
    transform_2022
)

# 2. Convert slope into risk
slope_risk = normalize_percentile(slope_2022)

# 3. Convert past erosion into risk
past_erosion = np.where(
    clean_change < 0,
    np.abs(clean_change),
    0
)

past_erosion_risk = normalize_percentile(past_erosion)

# 4. Redefine geomorph_risk for the GPS-adjusted crop
geomorph_risk = (
    0.55 * slope_risk +
    0.45 * past_erosion_risk
)

geomorph_risk = np.clip(geomorph_risk, 0, 1)

# 5. Confirm shape alignment
print("d22 shape:", d22.shape)
print("clean_change shape:", clean_change.shape)
print("slope_2022 shape:", slope_2022.shape)
print("geomorph_risk shape:", geomorph_risk.shape)

# 6. Visualize
plt.figure(figsize=(10, 8))
plt.imshow(geomorph_risk, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="Geomorphology risk score")
plt.title("GPS-Aligned Geomorphology Risk Map")
plt.axis("off")
plt.show()

In [ ]:
lichen_stress_surface = create_lichen_stress_surface(
    lichen_pixels,
    d22.shape,
    sigma=60
)

ecosignal_risk = (
    0.70 * geomorph_risk +
    0.30 * lichen_stress_surface
)

ecosignal_risk = np.clip(ecosignal_risk, 0, 1)

plt.figure(figsize=(10, 8))
plt.imshow(ecosignal_risk, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="EcoSignal future land-change risk")
plt.title("Aligned EcoSignal Risk Map")
plt.axis("off")
plt.show()

Linear DEM Generation

In [ ]:
years = [2022, 2027, 2032, 2037, 2042]

years_between = 2022 - 2015

annual_change_rate = clean_change / years_between

# Risk modifier:
# low risk = smaller projected change
# high risk = stronger projected change
risk_modifier = 0.5 + ecosignal_risk

future_dems = {}

for year in years:
    years_forward = year - 2022

    future_dems[year] = d22 + (
        annual_change_rate *
        years_forward *
        risk_modifier
    )

print("Generated future DEMs for:", list(future_dems.keys()))

Realistic(Exponential) DEM Generation

In [ ]:
years = [2022, 2027, 2032, 2037, 2042]

years_between = 2022 - 2015
annual_change_rate = clean_change / years_between

future_dems = {}

for year in years:
    years_forward = year - 2022

    # Nonlinear acceleration term
    # High EcoSignal risk causes change to accelerate over time
    acceleration = 1 + (ecosignal_risk * (years_forward / 20) ** 1.4)

    future_dems[year] = d22 + (
        annual_change_rate *
        years_forward *
        acceleration
    )

print("Generated nonlinear future DEMs:", list(future_dems.keys()))

In [ ]:
for year in years:
    predicted_change = future_dems[year] - d22

    plt.figure(figsize=(8, 6))
    plt.imshow(predicted_change, cmap="RdBu", vmin=-3.291, vmax=3.291)
    plt.colorbar(label="Predicted elevation change, meters")
    plt.title(f"Predicted Terrain Change: 2022–{year}")
    plt.axis("off")
    plt.show()

Diagnostic Cells

In [ ]:
for year in years:
    predicted_change = future_dems[year] - d22
    valid = np.isfinite(predicted_change)

    vals = predicted_change[valid]

    print("\nYear:", year)
    print("Min:", np.nanmin(vals))
    print("Max:", np.nanmax(vals))
    print("Mean absolute change:", np.nanmean(np.abs(vals)))
    print("95th percentile abs change:", np.nanpercentile(np.abs(vals), 95))
    print("99th percentile abs change:", np.nanpercentile(np.abs(vals), 99))
    print("% pixels > 0.15 m:", np.mean(np.abs(vals) > 0.15) * 100)
    print("% pixels > 1.0 m:", np.mean(np.abs(vals) > 1.0) * 100)

In [ ]:
final_change = future_dems[2042] - d22
valid = np.isfinite(final_change)

for p in [95, 98, 99, 99.5, 100]:
    scale = np.nanpercentile(np.abs(final_change[valid]), p)
    print(f"{p}th percentile absolute change: {scale:.3f} m")

In [ ]:
for year in years:
    dem = future_dems[year]
    change_from_2022 = dem - d22
    valid = np.isfinite(change_from_2022)

    print("\nYear:", year)
    print("Mean elevation:", np.nanmean(dem))
    print("Mean change from 2022:", np.nanmean(change_from_2022[valid]))
    print("Median change from 2022:", np.nanmedian(change_from_2022[valid]))
    print("Positive-change pixels %:", np.mean(change_from_2022[valid] > 0.15) * 100)
    print("Negative-change pixels %:", np.mean(change_from_2022[valid] < -0.15) * 100)

In [ ]:
valid = np.isfinite(clean_change)

positive_change = clean_change[valid & (clean_change > 0.15)]
negative_change = clean_change[valid & (clean_change < -0.15)]

print("Mean clean_change:", np.nanmean(clean_change[valid]))
print("Positive/deposition pixels:", len(positive_change))
print("Negative/erosion pixels:", len(negative_change))
print("Mean positive change:", np.nanmean(positive_change) if len(positive_change) else None)
print("Mean negative change:", np.nanmean(negative_change) if len(negative_change) else None)
print("Total deposition volume proxy:", np.nansum(np.where(clean_change > 0, clean_change, 0)))
print("Total erosion volume proxy:", np.nansum(np.where(clean_change < 0, clean_change, 0)))

In [ ]:
for year in years:
    dem = future_dems[year]
    change_from_2022 = dem - d22
    valid = np.isfinite(change_from_2022)

    print("\nYear:", year)
    print("Mean change from 2022:", np.nanmean(change_from_2022[valid]))
    print("Median change from 2022:", np.nanmedian(change_from_2022[valid]))
    print("Positive-change pixels %:", np.mean(change_from_2022[valid] > 0.15) * 100)
    print("Negative-change pixels %:", np.mean(change_from_2022[valid] < -0.15) * 100)
    print("Min change:", np.nanmin(change_from_2022[valid]))
    print("Max change:", np.nanmax(change_from_2022[valid]))

In [ ]:
for year in years:
    change_from_2022 = future_dems[year] - d22

    peak_changes = change_from_2022[peak_or_ridge_mask]
    valid_peak_changes = peak_changes[np.isfinite(peak_changes)]

    if len(valid_peak_changes) > 0:
        print("\nYear:", year)
        print("Mean change on peaks/ridges:", np.nanmean(valid_peak_changes))
        print("Max change on peaks/ridges:", np.nanmax(valid_peak_changes))
        print("% peak/ridge pixels increasing > 0.05m:",
              np.mean(valid_peak_changes > 0.05) * 100)

In [ ]:
for year in years:
    change_from_2022 = future_dems[year] - d22
    valid = np.isfinite(change_from_2022)

    deposition_pixels = change_from_2022 > 0.05
    erosion_pixels = change_from_2022 < -0.05

    print("\nYear:", year)
    print("Mean elevation of deposition pixels:",
          np.nanmean(d22[deposition_pixels]) if np.any(deposition_pixels) else None)
    print("Mean elevation of erosion pixels:",
          np.nanmean(d22[erosion_pixels]) if np.any(erosion_pixels) else None)
    print("Mean ridge score of deposition pixels:",
          np.nanmean(ridge_score[deposition_pixels]) if np.any(deposition_pixels) else None)
    print("Mean valley score of deposition pixels:",
          np.nanmean(valley_score[deposition_pixels]) if np.any(deposition_pixels) else None)
    print("Deposition pixels on peaks/ridges:",
          np.sum(deposition_pixels & peak_or_ridge_mask))

In [ ]:
# -----------------------------
# Final scientific QA checks
# -----------------------------

print("=== Shape checks ===")
print("d22:", d22.shape)
print("ecosignal_risk:", ecosignal_risk.shape)
print("erosion_susceptibility:", erosion_susceptibility.shape)
print("ridge_score:", ridge_score.shape)
print("valley_score:", valley_score.shape)

print("\n=== Peak/ridge deposition check ===")
for year in years:
    change = future_dems[year] - d22

    peak_gain = change[peak_or_ridge_mask]
    peak_gain = peak_gain[np.isfinite(peak_gain)]

    print(
        year,
        "max peak/ridge gain:",
        np.nanmax(peak_gain) if len(peak_gain) else None,
        "| % peak/ridge gain > 0.05m:",
        np.mean(peak_gain > 0.05) * 100 if len(peak_gain) else None
    )

print("\n=== Deposition terrain-position check ===")
for year in years:
    change = future_dems[year] - d22

    dep_pixels = change > 0.05
    ero_pixels = change < -0.05

    print("\nYear:", year)
    print("Deposition pixels:", np.sum(dep_pixels))
    print("Erosion pixels:", np.sum(ero_pixels))
    print("Deposition on peaks/ridges:", np.sum(dep_pixels & peak_or_ridge_mask))

    if np.any(dep_pixels):
        print("Mean deposition elevation:", np.nanmean(d22[dep_pixels]))
        print("Mean deposition valley score:", np.nanmean(valley_score[dep_pixels]))
        print("Mean deposition ridge score:", np.nanmean(ridge_score[dep_pixels]))

    if np.any(ero_pixels):
        print("Mean erosion elevation:", np.nanmean(d22[ero_pixels]))
        print("Mean erosion slope score:", np.nanmean(slope_score[ero_pixels]))
        print("Mean erosion EcoSignal risk:", np.nanmean(ecosignal_risk[ero_pixels]))

print("\n=== Magnitude check ===")
for year in years:
    change = future_dems[year] - d22
    valid = np.isfinite(change)

    print("\nYear:", year)
    print("Min change:", np.nanmin(change[valid]))
    print("Max change:", np.nanmax(change[valid]))
    print("Mean change:", np.nanmean(change[valid]))
    print("98th percentile abs change:", np.nanpercentile(np.abs(change[valid]), 98))

In [ ]:
print("=== Revised model QA ===")

for year in years:
    change = future_dems[year] - d22
    valid = np.isfinite(change)

    print("\nYear:", year)
    print("Min change:", np.nanmin(change[valid]))
    print("Max change:", np.nanmax(change[valid]))
    print("Mean change:", np.nanmean(change[valid]))
    print("98th percentile abs change:", np.nanpercentile(np.abs(change[valid]), 98))

    peak_change = change[peak_or_ridge_mask]
    peak_change = peak_change[np.isfinite(peak_change)]

    print("Max peak/ridge gain:", np.nanmax(peak_change) if len(peak_change) else None)
    print("% peak/ridge gain > 0.05m:",
          np.mean(peak_change > 0.05) * 100 if len(peak_change) else None)

    dep_pixels = change > 0.05
    ero_pixels = change < -0.05

    print("Deposition pixels:", np.sum(dep_pixels))
    print("Erosion pixels:", np.sum(ero_pixels))
    print("Deposition on peaks/ridges:", np.sum(dep_pixels & peak_or_ridge_mask))

    if np.any(dep_pixels):
        print("Mean deposition elevation:", np.nanmean(d22[dep_pixels]))
        print("Mean deposition valley score:", np.nanmean(valley_score[dep_pixels]))
        print("Mean deposition ridge score:", np.nanmean(ridge_score[dep_pixels]))

    if np.any(ero_pixels):
        print("Mean erosion elevation:", np.nanmean(d22[ero_pixels]))
        print("Mean erosion slope score:", np.nanmean(slope_score[ero_pixels]))
        print("Mean erosion EcoSignal risk:", np.nanmean(ecosignal_risk[ero_pixels]))

In [ ]:
for year in years:
    change = future_dems[year] - d22

    major_erosion = change < -0.25
    major_deposition = change > 0.15

    print("\nYear:", year)
    print("Major erosion pixels > 0.25m loss:", np.sum(major_erosion))
    print("Major deposition pixels > 0.15m gain:", np.sum(major_deposition))

    if np.any(major_erosion):
        print("Mean major erosion elevation:", np.nanmean(d22[major_erosion]))
        print("Mean major erosion slope score:", np.nanmean(slope_score[major_erosion]))
        print("Mean major erosion EcoSignal risk:", np.nanmean(ecosignal_risk[major_erosion]))
        print("Mean major erosion ridge score:", np.nanmean(ridge_score[major_erosion]))

    if np.any(major_deposition):
        print("Mean major deposition elevation:", np.nanmean(d22[major_deposition]))
        print("Mean major deposition valley score:", np.nanmean(valley_score[major_deposition]))
        print("Mean major deposition ridge score:", np.nanmean(ridge_score[major_deposition]))
        print("Major deposition on peaks/ridges:", np.sum(major_deposition & peak_or_ridge_mask))

Erosion-Focused Model

In [ ]:
years = [2022, 2027, 2032, 2037, 2042]

years_between = 2022 - 2015

# -----------------------------
# 1. Separate erosion and deposition
# -----------------------------

# Erosion/lowering signal only
erosion_change = np.where(clean_change < -lod, clean_change, 0)

# Deposition/raising signal kept separate
deposition_change = np.where(clean_change > lod, clean_change, 0)

# Convert to annual rates
annual_erosion_rate = np.abs(erosion_change) / years_between
annual_deposition_rate = deposition_change / years_between

# -----------------------------
# 2. Cap extreme rates
# -----------------------------
# This prevents pits, walls, water edges, construction artifacts, or DEM noise
# from dominating the future model.

valid_erosion = annual_erosion_rate[
    np.isfinite(annual_erosion_rate) &
    (annual_erosion_rate > 0)
]

erosion_rate_cap = np.nanpercentile(valid_erosion, 98)

annual_erosion_rate_capped = np.clip(
    annual_erosion_rate,
    0,
    erosion_rate_cap
)

print("Erosion rate cap, m/year:", erosion_rate_cap)

# -----------------------------
# 3. Build erosion susceptibility
# -----------------------------
# This is the actual EcoSignal prediction layer.
# Lichen stress affects erosion susceptibility, not deposition.

erosion_susceptibility = (
    0.65 * geomorph_risk +
    0.35 * lichen_stress_surface
)

erosion_susceptibility = np.clip(
    erosion_susceptibility,
    0,
    1
)

# -----------------------------
# 4. Generate future DEMs
# -----------------------------
# Main rule:
# future DEM = current DEM - erosion lowering
#
# Deposition is NOT biologically amplified.
# We include only a very small deposition background so the terrain does not
# become an unrealistic one-way erosion-only world.

future_dems = {}

for year in years:
    years_forward = year - 2022

    # Nonlinear acceleration:
    # high-risk areas erode faster over longer time horizons
    erosion_acceleration = 1 + (
        erosion_susceptibility *
        (years_forward / 20) ** 1.4
    )

    predicted_erosion_lowering = (
        annual_erosion_rate_capped *
        years_forward *
        erosion_acceleration
    )

    # Keep deposition weak and separate from lichen stress
    predicted_deposition_gain = (
        annual_deposition_rate *
        years_forward *
        0.10
    )

    future_dems[year] = (
        d22
        - predicted_erosion_lowering
        + predicted_deposition_gain
    )

print("Generated scientifically safer erosion-focused DEMs:", list(future_dems.keys()))

In [ ]:
final_change = future_dems[2042] - d22
valid = np.isfinite(final_change)

scale = np.nanpercentile(np.abs(final_change[valid]), 98)

print("Using shared color scale:", scale)

for year in years:
    predicted_change = future_dems[year] - d22

    plt.figure(figsize=(8, 6))
    plt.imshow(
        predicted_change,
        cmap="RdBu",
        vmin=-scale,
        vmax=scale
    )
    plt.colorbar(label="Predicted elevation change, meters")
    plt.title(f"EcoSignal Erosion-Focused Prediction: 2022–{year}")
    plt.axis("off")
    plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(erosion_susceptibility, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="Erosion susceptibility")
plt.title("EcoSignal Erosion Susceptibility: Geomorphology + Lichen Stress")
plt.axis("off")
plt.show()

Topographic-Position Layers for DEM

In [ ]:
from scipy.ndimage import uniform_filter
import numpy as np
import matplotlib.pyplot as plt

def normalize_percentile(arr, low=5, high=95):
    valid = np.isfinite(arr)
    p_low, p_high = np.nanpercentile(arr[valid], [low, high])

    if p_high == p_low:
        return np.zeros_like(arr, dtype="float32")

    norm = (arr - p_low) / (p_high - p_low)
    return np.clip(norm, 0, 1)


# -----------------------------
# 1. Relative elevation
# -----------------------------
elevation_norm = normalize_percentile(d22)

# -----------------------------
# 2. Topographic Position Index
# -----------------------------
# TPI > 0 = higher than local surroundings, ridge/peak tendency
# TPI < 0 = lower than local surroundings, valley/depression tendency

local_window = 101  # about 101 pixels; with 1m DEM, about 101m neighborhood

local_mean_elev = uniform_filter(
    np.nan_to_num(d22, nan=np.nanmean(d22)),
    size=local_window
)

tpi = d22 - local_mean_elev

ridge_score = normalize_percentile(tpi)
valley_score = normalize_percentile(-tpi)

# -----------------------------
# 3. Slope score
# -----------------------------
slope_score = normalize_percentile(slope_2022)

plt.figure(figsize=(10, 8))
plt.imshow(ridge_score, cmap="magma", vmin=0, vmax=1)
plt.colorbar(label="Ridge / peak score")
plt.title("Ridge and Peak Score")
plt.axis("off")
plt.show()

plt.figure(figsize=(10, 8))
plt.imshow(valley_score, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Valley / low-position score")
plt.title("Valley and Depression Score")
plt.axis("off")
plt.show()

In [ ]:
# -----------------------------
# Peak / ridge mask
# -----------------------------
# These areas should NOT gain elevation in the future model.
# They are high relative elevation and locally ridge-like.

peak_or_ridge_mask = (
    (elevation_norm > 0.70) &
    (ridge_score > 0.60)
)

# -----------------------------
# Deposition eligibility
# -----------------------------
# Deposition is only allowed in low/receiving areas:
# - valley-like
# - not steep
# - not a ridge or peak

deposition_allowed = (
    (valley_score > 0.65) &
    (slope_score < 0.35) &
    (elevation_norm < 0.55) &
    (~peak_or_ridge_mask)
)

# -----------------------------
# Erosion eligibility
# -----------------------------
# Erosion is favored on:
# - steep areas
# - ridge/convex areas
# - areas with high EcoSignal risk

erosion_allowed = (
    (slope_score > 0.35) |
    (ridge_score > 0.55) |
    (ecosignal_risk > 0.55)
)

plt.figure(figsize=(10, 8))
plt.imshow(peak_or_ridge_mask, cmap="gray")
plt.title("Peak/Ridge Mask: Deposition Blocked Here")
plt.axis("off")
plt.show()

plt.figure(figsize=(10, 8))
plt.imshow(deposition_allowed, cmap="Blues")
plt.title("Deposition-Allowed Areas")
plt.axis("off")
plt.show()

plt.figure(figsize=(10, 8))
plt.imshow(erosion_allowed, cmap="Reds")
plt.title("Erosion-Allowed Areas")
plt.axis("off")
plt.show()

In [ ]:
years = [2022, 2027, 2032, 2037, 2042]

years_between = 2022 - 2015

# -----------------------------
# Separate observed past change
# -----------------------------
raw_erosion_change = np.where(clean_change < -lod, clean_change, 0)
raw_deposition_change = np.where(clean_change > lod, clean_change, 0)

# -----------------------------
# Apply terrain-position constraints
# -----------------------------
# Erosion only where erosion is physically plausible
erosion_change = np.where(
    erosion_allowed,
    raw_erosion_change,
    0
)

# Deposition only where deposition is physically plausible
deposition_change = np.where(
    deposition_allowed,
    raw_deposition_change,
    0
)

# Absolutely block deposition on peaks/ridges
deposition_change = np.where(
    peak_or_ridge_mask,
    0,
    deposition_change
)

# Convert to annual rates
annual_erosion_rate = np.abs(erosion_change) / years_between
annual_deposition_rate = deposition_change / years_between

# -----------------------------
# Cap extreme rates
# -----------------------------
valid_erosion = annual_erosion_rate[
    np.isfinite(annual_erosion_rate) &
    (annual_erosion_rate > 0)
]

valid_deposition = annual_deposition_rate[
    np.isfinite(annual_deposition_rate) &
    (annual_deposition_rate > 0)
]

erosion_rate_cap = np.nanpercentile(valid_erosion, 98) if len(valid_erosion) else 0
deposition_rate_cap = np.nanpercentile(valid_deposition, 95) if len(valid_deposition) else 0

annual_erosion_rate_capped = np.clip(
    annual_erosion_rate,
    0,
    erosion_rate_cap
)

annual_deposition_rate_capped = np.clip(
    annual_deposition_rate,
    0,
    deposition_rate_cap
)

print("Erosion rate cap, m/year:", erosion_rate_cap)
print("Deposition rate cap, m/year:", deposition_rate_cap)

# -----------------------------
# Erosion susceptibility
# -----------------------------
# Lichen stress modifies erosion susceptibility only.
erosion_susceptibility = (
    0.60 * geomorph_risk +
    0.25 * lichen_stress_surface +
    0.15 * slope_score
)

erosion_susceptibility = np.clip(erosion_susceptibility, 0, 1)

# -----------------------------
# Generate future DEMs
# -----------------------------
future_dems = {}

for year in years:
    years_forward = year - 2022

    # Erosion accelerates in high-risk areas
    erosion_acceleration = 1 + (
        erosion_susceptibility *
        (years_forward / 20) ** 1.4
    )

    predicted_erosion_lowering = (
        annual_erosion_rate_capped *
        years_forward *
        erosion_acceleration
    )

    # Deposition is conservative and not lichen-amplified.
    # It is also limited to terrain-position-allowed areas.
    predicted_deposition_gain = (
        annual_deposition_rate_capped *
        years_forward *
        0.15
    )

    # Hard block: no deposition on peaks/ridges
    predicted_deposition_gain = np.where(
        peak_or_ridge_mask,
        0,
        predicted_deposition_gain
    )

    future_dems[year] = (
        d22
        - predicted_erosion_lowering
        + predicted_deposition_gain
    )

print("Generated terrain-constrained future DEMs:", list(future_dems.keys()))

Deposition Cap

In [ ]:
years = [2022, 2027, 2032, 2037, 2042]

years_between = 2022 - 2015

# Separate observed past change
raw_erosion_change = np.where(clean_change < -lod, clean_change, 0)
raw_deposition_change = np.where(clean_change > lod, clean_change, 0)

# Apply terrain-position constraints
erosion_change = np.where(
    erosion_allowed,
    raw_erosion_change,
    0
)

deposition_change = np.where(
    deposition_allowed,
    raw_deposition_change,
    0
)

# Absolutely block deposition on peaks/ridges
deposition_change = np.where(
    peak_or_ridge_mask,
    0,
    deposition_change
)

# Convert to annual rates
annual_erosion_rate = np.abs(erosion_change) / years_between
annual_deposition_rate = deposition_change / years_between

# Percentile caps
valid_erosion = annual_erosion_rate[
    np.isfinite(annual_erosion_rate) &
    (annual_erosion_rate > 0)
]

valid_deposition = annual_deposition_rate[
    np.isfinite(annual_deposition_rate) &
    (annual_deposition_rate > 0)
]

erosion_rate_cap = np.nanpercentile(valid_erosion, 95) if len(valid_erosion) else 0
deposition_rate_cap = np.nanpercentile(valid_deposition, 90) if len(valid_deposition) else 0

annual_erosion_rate_capped = np.clip(
    annual_erosion_rate,
    0,
    erosion_rate_cap
)

annual_deposition_rate_capped = np.clip(
    annual_deposition_rate,
    0,
    deposition_rate_cap
)

print("Conservative erosion rate cap, m/year:", erosion_rate_cap)
print("Conservative deposition rate cap, m/year:", deposition_rate_cap)

# Erosion susceptibility
erosion_susceptibility = (
    0.60 * geomorph_risk +
    0.25 * lichen_stress_surface +
    0.15 * slope_score
)

erosion_susceptibility = np.clip(erosion_susceptibility, 0, 1)

# Hard scenario caps by final year
max_total_erosion_2042 = 2.0      # meters
max_total_deposition_2042 = 0.75  # meters

future_dems = {}

for year in years:
    years_forward = year - 2022
    scenario_fraction = years_forward / (2042 - 2022) if year != 2022 else 0

    # Risk-accelerated erosion
    erosion_acceleration = 1 + (
        erosion_susceptibility *
        (years_forward / 20) ** 1.2
    )

    predicted_erosion_lowering = (
        annual_erosion_rate_capped *
        years_forward *
        erosion_acceleration
    )

    predicted_deposition_gain = (
        annual_deposition_rate_capped *
        years_forward *
        0.15
    )

    # Hard cap total predicted change by year
    predicted_erosion_lowering = np.clip(
        predicted_erosion_lowering,
        0,
        max_total_erosion_2042 * scenario_fraction
    )

    predicted_deposition_gain = np.clip(
        predicted_deposition_gain,
        0,
        max_total_deposition_2042 * scenario_fraction
    )

    # No deposition on peaks/ridges
    predicted_deposition_gain = np.where(
        peak_or_ridge_mask,
        0,
        predicted_deposition_gain
    )

    future_dems[year] = (
        d22
        - predicted_erosion_lowering
        + predicted_deposition_gain
    )

print("Generated final conservative terrain-constrained future DEMs:", list(future_dems.keys()))

Realistic V2 modifying extremes by adjusting prediction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

years = [2022, 2027, 2032, 2037, 2042]
years_between = 2022 - 2015

# -----------------------------
# 1. Separate observed past change
# -----------------------------

past_erosion_mag = np.where(
    clean_change < -lod,
    np.abs(clean_change),
    0
)

past_deposition_mag = np.where(
    clean_change > lod,
    clean_change,
    0
)

# -----------------------------
# 2. Normalize past evidence
# -----------------------------
# Past DEM difference becomes evidence, not direct future rate.

def normalize_percentile(arr, low=5, high=95):
    valid = np.isfinite(arr)
    p_low, p_high = np.nanpercentile(arr[valid], [low, high])

    if p_high == p_low:
        return np.zeros_like(arr, dtype="float32")

    norm = (arr - p_low) / (p_high - p_low)
    return np.clip(norm, 0, 1)

past_erosion_evidence = normalize_percentile(past_erosion_mag, low=50, high=98)
past_deposition_evidence = normalize_percentile(past_deposition_mag, low=50, high=98)

# Smooth evidence slightly so isolated single-pixel artifacts do not dominate
past_erosion_evidence = gaussian_filter(past_erosion_evidence, sigma=2)
past_deposition_evidence = gaussian_filter(past_deposition_evidence, sigma=2)

past_erosion_evidence = np.clip(past_erosion_evidence, 0, 1)
past_deposition_evidence = np.clip(past_deposition_evidence, 0, 1)

# -----------------------------
# 3. Build physically plausible erosion potential
# -----------------------------
# Erosion should be strongest on slopes/ridges/high-risk zones,
# and strengthened by lichen stress.

erosion_potential = (
    0.30 * slope_score +
    0.20 * ridge_score +
    0.20 * past_erosion_evidence +
    0.20 * ecosignal_risk +
    0.10 * lichen_stress_surface
)

erosion_potential = np.clip(erosion_potential, 0, 1)

# Erosion only where physically plausible
erosion_potential = np.where(
    erosion_allowed,
    erosion_potential,
    0
)

# -----------------------------
# 4. Convert erosion potential to annual erosion rate
# -----------------------------
# This is the key change.
# We use a realistic modeled rate instead of raw DEM-change continuation.

min_erosion_rate = 0.000   # m/year
max_erosion_rate = 0.080   # m/year for high-risk cells

annual_erosion_rate_modeled = (
    min_erosion_rate +
    (max_erosion_rate - min_erosion_rate) * (erosion_potential ** 1.7)
)

# Remove erosion in very low-risk cells
annual_erosion_rate_modeled = np.where(
    erosion_potential > 0.25,
    annual_erosion_rate_modeled,
    0
)

# -----------------------------
# 5. Build physically plausible deposition potential
# -----------------------------
# Deposition should be restricted to receiving zones:
# valley-like, flatter, lower relative position.
# Past deposition evidence helps, but does not dominate.

receiving_zone_score = (
    0.45 * valley_score +
    0.30 * (1 - slope_score) +
    0.15 * (1 - elevation_norm) +
    0.10 * past_deposition_evidence
)

receiving_zone_score = np.clip(receiving_zone_score, 0, 1)

deposition_potential = receiving_zone_score * deposition_allowed

# Deposition should not occur on peaks/ridges
deposition_potential = np.where(
    peak_or_ridge_mask,
    0,
    deposition_potential
)

# Only allow deposition in strong receiving zones
deposition_potential = np.where(
    deposition_potential > 0.68,
    deposition_potential,
    0
)

max_deposition_rate = 0.018

annual_deposition_rate_modeled = (
    max_deposition_rate * (deposition_potential ** 2.0)
)

# -----------------------------
# 7. Generate future DEMs
# -----------------------------

future_dems = {}

for year in years:
    years_forward = year - 2022

    # Erosion accelerates mildly in high-risk zones
    erosion_time_factor = 1 + (
        0.35 * erosion_potential * (years_forward / 20)
    )

    # Deposition grows more slowly and is not lichen-amplified
    deposition_time_factor = 1 + (
        0.10 * deposition_potential * (years_forward / 20)
    )

    predicted_erosion_lowering = (
        annual_erosion_rate_modeled *
        years_forward *
        erosion_time_factor
    )

    predicted_deposition_gain = (
        annual_deposition_rate_modeled *
        years_forward *
        deposition_time_factor
    )

    # Safety rule only, not the main model:
    # no deposition on ridges/peaks
    predicted_deposition_gain = np.where(
        peak_or_ridge_mask,
        0,
        predicted_deposition_gain
    )

    future_dems[year] = (
        d22
        - predicted_erosion_lowering
        + predicted_deposition_gain
    )

print("Generated modeled terrain-constrained future DEMs:", list(future_dems.keys()))

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(erosion_potential, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="Erosion potential")
plt.title("EcoSignal Modeled Erosion Potential")
plt.axis("off")
plt.show()

plt.figure(figsize=(10, 8))
plt.imshow(deposition_potential, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Deposition potential")
plt.title("Terrain-Constrained Deposition Potential")
plt.axis("off")
plt.show()

2D Map

In [ ]:
final_change = future_dems[2042] - d22
valid = np.isfinite(final_change)

scale = np.nanpercentile(np.abs(final_change[valid]), 98)

print("Using shared color scale:", scale)

for year in years:
    predicted_change = future_dems[year] - d22

    plt.figure(figsize=(8, 6))
    plt.imshow(
        predicted_change,
        cmap="RdBu",
        vmin=-scale,
        vmax=scale
    )
    plt.colorbar(label="Predicted elevation change, meters")
    plt.title(f"EcoSignal Terrain-Constrained Scenario: 2022–{year}")
    plt.axis("off")
    plt.show()

3D Time-Series Map

In [ ]:
step = 10

z_by_year = {
    year: future_dems[year][::step, ::step]
    for year in years
}

risk_downsampled = ecosignal_risk[::step, ::step]

rows, cols = z_by_year[2022].shape
x = np.arange(cols)
y = np.arange(rows)

# Lichen points for 3D plot
points_for_plot = lichen_pixels[
    lichen_pixels["inside_crop"] == True
].copy()

point_rows = points_for_plot["row"].astype(int).values
point_cols = points_for_plot["col"].astype(int).values

plot_x = point_cols / step
plot_y = point_rows / step

plot_z = []

for r, c in zip(point_rows, point_cols):
    plot_z.append(d22[r, c] + 1.0)

plot_z = np.array(plot_z)

fig = go.Figure()

# Initial surface: 2022
fig.add_trace(go.Surface(
    z=z_by_year[2022],
    x=x,
    y=y,
    surfacecolor=risk_downsampled,
    colorscale="Inferno",
    cmin=0,
    cmax=1,
    colorbar=dict(title="EcoSignal risk"),
    showscale=True
))

# Lichen observation points
fig.add_trace(go.Scatter3d(
    x=plot_x,
    y=plot_y,
    z=plot_z,
    mode="markers",
    marker=dict(
        size=6,
        color=points_for_plot["lichen_stress"],
        colorscale="YlOrRd",
        cmin=0,
        cmax=1,
        colorbar=dict(title="Lichen stress")
    ),
    text=[
        f"Lat: {lat:.5f}<br>"
        f"Lon: {lon:.5f}<br>"
        f"Lichen stress: {stress:.2f}<br>"
        f"Confidence: {conf:.2f}"
        for lat, lon, stress, conf in zip(
            points_for_plot["latitude"],
            points_for_plot["longitude"],
            points_for_plot["lichen_stress"],
            points_for_plot["cv_confidence"]
        )
    ],
    hoverinfo="text",
    name="Lichen observations"
))

# Animation frames
frames = []

for year in years:
    frames.append(go.Frame(
        name=str(year),
        data=[
            go.Surface(
                z=z_by_year[year],
                x=x,
                y=y,
                surfacecolor=risk_downsampled,
                colorscale="Inferno",
                cmin=0,
                cmax=1,
                showscale=True
            )
        ],
        traces=[0]
    ))

fig.frames = frames

# Slider controls
sliders = [{
    "active": 0,
    "currentvalue": {
        "prefix": "Year: ",
        "font": {"size": 18}
    },
    "steps": [
        {
            "label": str(year),
            "method": "animate",
            "args": [
                [str(year)],
                {
                    "mode": "immediate",
                    "frame": {"duration": 700, "redraw": True},
                    "transition": {"duration": 300}
                }
            ]
        }
        for year in years
    ]
}]

# Play / pause buttons
updatemenus = [{
    "type": "buttons",
    "showactive": False,
    "buttons": [
        {
            "label": "Play",
            "method": "animate",
            "args": [
                None,
                {
                    "frame": {"duration": 700, "redraw": True},
                    "fromcurrent": True,
                    "transition": {"duration": 300}
                }
            ]
        },
        {
            "label": "Pause",
            "method": "animate",
            "args": [
                [None],
                {
                    "frame": {"duration": 0, "redraw": False},
                    "mode": "immediate",
                    "transition": {"duration": 0}
                }
            ]
        }
    ]
}]

# Use the real, scientifically accurate DEM values
z_by_year = {
    year: future_dems[year][::step, ::step]
    for year in years
}

all_z_values = np.concatenate([
    future_dems[year][np.isfinite(future_dems[year])].ravel()
    for year in years
])

z_min = np.nanpercentile(all_z_values, 1)
z_max = np.nanpercentile(all_z_values, 99)

print("Fixed z range:", z_min, z_max)

fig.update_layout(
    title="EcoSignal Future Landscape Simulation: 2022–2042",
    scene=dict(
        xaxis_title="X pixel",
        yaxis_title="Y pixel",
        zaxis_title="Predicted elevation",
        aspectmode="manual",
        zaxis=dict(range=[z_min, z_max]),
        aspectratio=dict(x=1, y=1, z=0.75)
    ),
    sliders=sliders,
    updatemenus=updatemenus,
    width=1050,
    height=850
)

fig.show()

Lichen Contribution to Map Plot

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np

final_change = future_dems[2042] - d22
valid = np.isfinite(final_change)

scale = np.nanpercentile(np.abs(final_change[valid]), 98)

print("Using shared color scale:", scale)

norm = TwoSlopeNorm(vmin=-scale, vcenter=0, vmax=scale)

for year in years:
    predicted_change = future_dems[year] - d22

    plt.figure(figsize=(8, 6))
    plt.imshow(
        predicted_change,
        cmap="RdBu",
        norm=norm
    )
    plt.colorbar(label="Predicted elevation change, meters")
    plt.title(f"EcoSignal Modeled Terrain Scenario: 2022–{year}")
    plt.axis("off")
    plt.show()

Ablation Tests

In [ ]:
# Lichen-only risk model
lichen_only_risk = lichen_stress_surface.copy()
lichen_only_risk = np.clip(lichen_only_risk, 0, 1)

plt.figure(figsize=(10, 8))
plt.imshow(lichen_only_risk, cmap="YlOrRd", vmin=0, vmax=1)
plt.colorbar(label="Lichen-only stress risk")
plt.title("Lichen-Only Ablation Risk Map")
plt.axis("off")
plt.show()

In [ ]:
# DEM-only model: no lichen input
dem_only_risk = (
    0.45 * slope_score +
    0.25 * ridge_score +
    0.20 * past_erosion_evidence +
    0.10 * valley_score
)

dem_only_risk = np.clip(dem_only_risk, 0, 1)

plt.figure(figsize=(10, 8))
plt.imshow(dem_only_risk, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="DEM-only geomorphology risk")
plt.title("DEM-Only Ablation Risk Map")
plt.axis("off")
plt.show()

In [ ]:
combined_risk = erosion_potential.copy()

plt.figure(figsize=(10, 8))
plt.imshow(combined_risk, cmap="inferno", vmin=0, vmax=1)
plt.colorbar(label="Combined EcoSignal risk")
plt.title("Combined EcoSignal Risk Map")
plt.axis("off")
plt.show()

In [ ]:
def top_risk_overlap_score(risk_map, observed_erosion, top_percent=10):
    valid = np.isfinite(risk_map) & np.isfinite(observed_erosion)

    threshold = np.nanpercentile(risk_map[valid], 100 - top_percent)
    top_risk = risk_map >= threshold

    overlap = top_risk & observed_erosion & valid

    score = np.sum(overlap) / np.sum(top_risk & valid)

    return score

observed_erosion = clean_change < -lod

dem_score = top_risk_overlap_score(
    dem_only_risk,
    observed_erosion,
    top_percent=10
)

lichen_score = top_risk_overlap_score(
    lichen_only_risk,
    observed_erosion,
    top_percent=10
)

combined_score = top_risk_overlap_score(
    combined_risk,
    observed_erosion,
    top_percent=10
)

print("Top 10% risk overlap with observed erosion:")
print("DEM-only:", dem_score)
print("Lichen-only:", lichen_score)
print("Combined EcoSignal:", combined_score)

Hindcasting Scientific Validation

In [ ]:
from pathlib import Path

DEM_DIR = Path("/content/drive/MyDrive/EcoSignal_DEM")

dem_2013_path = DEM_DIR / "Dem_2013.tif"
dem_2015_path = DEM_DIR / "Dem_2015.tif"
dem_2022_path = DEM_DIR / "Dem_2022.tif"
dem_diff_path = DEM_DIR / "DemDif_2015-2022.tif"

for p in [dem_2013_path, dem_2015_path, dem_2022_path]:
    print(p.name, "exists:", p.exists())

In [ ]:
import rasterio
import numpy as np

from rasterio.windows import from_bounds, Window
from rasterio.warp import (
    transform_bounds,
    reproject,
    Resampling
)
from rasterio.transform import array_bounds

In [ ]:
def read_reference_crop(path, bounds_wgs84):
    """
    Creates the reference raster crop using WGS84 GPS bounds.
    """

    with rasterio.open(path) as src:

        raster_bounds = transform_bounds(
            "EPSG:4326",
            src.crs,
            *bounds_wgs84,
            densify_pts=21
        )

        window = from_bounds(
            *raster_bounds,
            transform=src.transform
        )

        window = window.round_offsets().round_lengths()

        full_window = Window(
            0, 0,
            src.width,
            src.height
        )

        window = window.intersection(full_window)

        data = src.read(
            1,
            window=window,
            masked=True
        ).filled(np.nan).astype("float32")

        transform = src.window_transform(window)

        profile = src.profile.copy()

        profile.update({
            "height": data.shape[0],
            "width": data.shape[1],
            "transform": transform
        })

        return data, transform, profile

In [ ]:
d15_val, ref_transform, ref_profile = read_reference_crop(
    dem_2015_path,
    crop_bounds_wgs84
)

ref_crs = ref_profile["crs"]
ref_shape = d15_val.shape

print("Reference shape:", ref_shape)
print("Reference CRS:", ref_crs)
print("Reference transform:")
print(ref_transform)

In [ ]:
def align_dem_to_reference(
    source_path,
    ref_shape,
    ref_transform,
    ref_crs
):
    """
    Align another DEM exactly to the reference raster grid.
    """

    destination = np.full(
        ref_shape,
        np.nan,
        dtype="float32"
    )

    # Geographic extent required by destination
    left, bottom, right, top = array_bounds(
        ref_shape[0],
        ref_shape[1],
        ref_transform
    )

    with rasterio.open(source_path) as src:

        source_bounds = transform_bounds(
            ref_crs,
            src.crs,
            left,
            bottom,
            right,
            top,
            densify_pts=21
        )

        source_window = from_bounds(
            *source_bounds,
            transform=src.transform
        )

        source_window = source_window.round_offsets().round_lengths()

        full_window = Window(
            0, 0,
            src.width,
            src.height
        )

        source_window = source_window.intersection(full_window)

        source = src.read(
            1,
            window=source_window,
            masked=True
        ).filled(np.nan).astype("float32")

        source_transform = src.window_transform(
            source_window
        )

        reproject(
            source=source,
            destination=destination,
            src_transform=source_transform,
            src_crs=src.crs,
            src_nodata=np.nan,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear
        )

    return destination

In [ ]:
d13_val = align_dem_to_reference(
    dem_2013_path,
    ref_shape,
    ref_transform,
    ref_crs
)

d22_val = align_dem_to_reference(
    dem_2022_path,
    ref_shape,
    ref_transform,
    ref_crs
)

print("2013:", d13_val.shape)
print("2015:", d15_val.shape)
print("2022:", d22_val.shape)

In [ ]:
dod_2013_2015 = d15_val - d13_val
dod_2015_2022 = d22_val - d15_val

print(
    "2013–2015 mean change:",
    np.nanmean(dod_2013_2015)
)

print(
    "2015–2022 mean change:",
    np.nanmean(dod_2015_2022)
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(
    dod_2013_2015,
    cmap="RdBu",
    vmin=-1,
    vmax=1
)

plt.colorbar(
    im,
    ax=ax,
    label="Elevation change (m)"
)

ax.set_title("Observed Terrain Change: 2013–2015")
ax.axis("off")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(
    dod_2015_2022,
    cmap="RdBu",
    vmin=-1,
    vmax=1
)

plt.colorbar(
    im,
    ax=ax,
    label="Elevation change (m)"
)

ax.set_title("Observed Terrain Change: 2015–2022")
ax.axis("off")

plt.show()

In [ ]:
from scipy.ndimage import (
    uniform_filter,
    distance_transform_edt
)

In [ ]:
def fill_nan_nearest(arr):
    arr = arr.copy()

    missing = ~np.isfinite(arr)

    if not np.any(missing):
        return arr

    indices = distance_transform_edt(
        missing,
        return_distances=False,
        return_indices=True
    )

    return arr[tuple(indices)]

In [ ]:
def normalize_array(arr, low=5, high=95):

    valid = np.isfinite(arr)

    p_low, p_high = np.nanpercentile(
        arr[valid],
        [low, high]
    )

    if p_high == p_low:
        return np.zeros_like(arr)

    output = (
        arr - p_low
    ) / (
        p_high - p_low
    )

    return np.clip(output, 0, 1)

In [ ]:
def build_terrain_features(
    dem,
    transform,
    tpi_window=101,
    roughness_window=15
):

    filled = fill_nan_nearest(dem)

    xres = abs(transform.a)
    yres = abs(transform.e)

    # ----------------------
    # Slope
    # ----------------------

    dz_dy, dz_dx = np.gradient(
        filled,
        yres,
        xres
    )

    slope = np.degrees(
        np.arctan(
            np.sqrt(
                dz_dx**2 +
                dz_dy**2
            )
        )
    )

    # ----------------------
    # TPI
    # ----------------------

    local_mean = uniform_filter(
        filled,
        size=tpi_window
    )

    tpi = filled - local_mean

    # ----------------------
    # Roughness
    # ----------------------

    mean_small = uniform_filter(
        filled,
        size=roughness_window
    )

    mean_sq = uniform_filter(
        filled**2,
        size=roughness_window
    )

    variance = np.maximum(
        mean_sq - mean_small**2,
        0
    )

    roughness = np.sqrt(variance)

    # ----------------------
    # Normalize
    # ----------------------

    elevation_norm = normalize_array(filled)

    slope_score = normalize_array(slope)

    ridge_score = normalize_array(tpi)

    valley_score = normalize_array(-tpi)

    roughness_score = normalize_array(roughness)

    feature_stack = np.stack([
        elevation_norm,
        slope_score,
        ridge_score,
        valley_score,
        roughness_score
    ], axis=-1)

    feature_names = [
        "elevation",
        "slope",
        "ridge",
        "valley",
        "roughness"
    ]

    return feature_stack, feature_names

In [ ]:
X_2013_map, feature_names = build_terrain_features(
    d13_val,
    ref_transform
)

X_2015_map, _ = build_terrain_features(
    d15_val,
    ref_transform
)

print("Feature shape:", X_2013_map.shape)
print("Features:", feature_names)

In [ ]:
LOD_TOTAL = 0.15

# 3 cm/year = initial sensitivity threshold.
EROSION_RATE_THRESHOLD = 0.03

In [ ]:
def create_erosion_labels(
    dod,
    years_between,
    lod_total=0.15,
    erosion_rate_threshold=0.03
):

    annual_rate = dod / years_between

    erosion = (
        (dod < -lod_total) &
        (annual_rate < -erosion_rate_threshold)
    )

    stable = (
        np.abs(dod) <= lod_total
    )

    # Ignore strong deposition cells during erosion training
    eligible = erosion | stable

    labels = np.full(
        dod.shape,
        -1,
        dtype="int8"
    )

    labels[stable] = 0
    labels[erosion] = 1

    return labels, eligible, annual_rate

In [ ]:
y_train_map, train_eligible, rate_13_15 = create_erosion_labels(
    dod_2013_2015,
    years_between=2,
    lod_total=LOD_TOTAL,
    erosion_rate_threshold=EROSION_RATE_THRESHOLD
)

y_test_map, test_eligible, rate_15_22 = create_erosion_labels(
    dod_2015_2022,
    years_between=7,
    lod_total=LOD_TOTAL,
    erosion_rate_threshold=EROSION_RATE_THRESHOLD
)

print(
    "Training erosion pixels:",
    np.sum(y_train_map == 1)
)

print(
    "Training stable pixels:",
    np.sum(y_train_map == 0)
)

print(
    "Testing erosion pixels:",
    np.sum(y_test_map == 1)
)

print(
    "Testing stable pixels:",
    np.sum(y_test_map == 0)
)

In [ ]:
!pip install scikit-learn -q

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve
)

In [ ]:
feature_valid_2013 = np.all(
    np.isfinite(X_2013_map),
    axis=-1
)

valid_train = (
    train_eligible &
    feature_valid_2013 &
    np.isfinite(d13_val) &
    np.isfinite(d15_val)
)

X_train_all = X_2013_map[valid_train]
y_train_all = y_train_map[valid_train]

print("Available training pixels:", len(y_train_all))
print("Training erosion prevalence:", np.mean(y_train_all))

In [ ]:
rng = np.random.default_rng(42)

MAX_TRAIN_PIXELS = 250_000

if len(y_train_all) > MAX_TRAIN_PIXELS:

    selected = rng.choice(
        len(y_train_all),
        size=MAX_TRAIN_PIXELS,
        replace=False
    )

    X_train = X_train_all[selected]
    y_train = y_train_all[selected]

else:
    X_train = X_train_all
    y_train = y_train_all

print("Pixels used for training:", len(y_train))

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
    oob_score=True
)

rf.fit(
    X_train,
    y_train
)

print("OOB accuracy:", rf.oob_score_)

In [ ]:
oob_probability = rf.oob_decision_function_[:, 1]

valid_oob = np.isfinite(oob_probability)

precision, recall, thresholds = precision_recall_curve(
    y_train[valid_oob],
    oob_probability[valid_oob]
)

f1_values = (
    2 * precision * recall /
    (precision + recall + 1e-12)
)

best_index = np.nanargmax(
    f1_values[:-1]
)

decision_threshold = thresholds[
    best_index
]

print(
    "Training-derived decision threshold:",
    decision_threshold
)

print(
    "Training OOB best F1:",
    f1_values[best_index]
)

In [ ]:
def predict_probability_batches(
    model,
    X,
    batch_size=100_000
):

    output = np.empty(
        len(X),
        dtype="float32"
    )

    for start in range(
        0,
        len(X),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(X)
        )

        output[start:end] = model.predict_proba(
            X[start:end]
        )[:, 1]

    return output

In [ ]:
feature_valid_2015 = np.all(
    np.isfinite(X_2015_map),
    axis=-1
)

valid_test = (
    test_eligible &
    feature_valid_2015 &
    np.isfinite(d15_val) &
    np.isfinite(d22_val)
)

X_test = X_2015_map[valid_test]
y_test = y_test_map[valid_test]

print("Testing pixels:", len(y_test))
print("Actual erosion prevalence:", np.mean(y_test))

In [ ]:
test_probability = predict_probability_batches(
    rf,
    X_test
)

test_prediction = (
    test_probability >= decision_threshold
).astype("int8")

In [ ]:
roc_auc = roc_auc_score(
    y_test,
    test_probability
)

pr_auc = average_precision_score(
    y_test,
    test_probability
)

precision_value = precision_score(
    y_test,
    test_prediction,
    zero_division=0
)

recall_value = recall_score(
    y_test,
    test_prediction,
    zero_division=0
)

f1_value = f1_score(
    y_test,
    test_prediction,
    zero_division=0
)

cm = confusion_matrix(
    y_test,
    test_prediction
)

print("=== TRUE TEMPORAL HINDCAST ===")
print()
print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))
print("Precision:", round(precision_value, 4))
print("Recall:", round(recall_value, 4))
print("F1:", round(f1_value, 4))
print()
print("Confusion matrix:")
print(cm)

In [ ]:
top_threshold = np.percentile(
    test_probability,
    90
)

top_risk = (
    test_probability >= top_threshold
)

top_risk_precision = np.mean(
    y_test[top_risk]
)

baseline_erosion_rate = np.mean(
    y_test
)

lift = (
    top_risk_precision /
    baseline_erosion_rate
)

erosion_captured = (
    np.sum(
        (y_test == 1) &
        top_risk
    ) /
    np.sum(y_test == 1)
)

print("=== TOP 10% RISK TEST ===")
print()
print(
    "Baseline erosion rate:",
    round(baseline_erosion_rate, 4)
)

print(
    "Actual erosion inside top 10%:",
    round(top_risk_precision, 4)
)

print(
    "Risk enrichment/lift:",
    round(lift, 2),
    "x"
)

print(
    "% of all observed erosion captured by top 10%:",
    round(erosion_captured * 100, 2),
    "%"
)

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# Ensure we project or crop the lichen_stress_surface to the same shape as the GPS-aligned validation grid
if lichen_stress_surface.shape != test_eligible.shape:
    # Option 1: If using synthetic test pixels, regenerate the surface using the ref_shape
    test_lichen_pixels_val = create_test_lichen_points(ref_shape)
    lichen_stress_surface_val = create_lichen_stress_surface(
        test_lichen_pixels_val,
        ref_shape,
        sigma=60
    )
else:
    lichen_stress_surface_val = lichen_stress_surface.copy()

lichen_association_map = lichen_stress_surface_val.copy()

association_valid = (
    np.isfinite(lichen_association_map) &
    test_eligible
)

lichen_scores = lichen_association_map[
    association_valid
]

past_erosion_labels = y_test_map[
    association_valid
]

if len(np.unique(past_erosion_labels)) == 2:
    lichen_auc = roc_auc_score(
        past_erosion_labels,
        lichen_scores
    )

    lichen_pr_auc = average_precision_score(
        past_erosion_labels,
        lichen_scores
    )

    print("=== LICHEN RETROSPECTIVE ASSOCIATION ===")
    print("ROC-AUC:", round(lichen_auc, 4))
    print("PR-AUC:", round(lichen_pr_auc, 4))
else:
    print("Not enough erosion/stable classes for association test.")

In [ ]:
prediction_probability_map = np.full(
    d15_val.shape,
    np.nan,
    dtype="float32"
)

prediction_probability_map[
    valid_test
] = test_probability

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(
    prediction_probability_map,
    cmap="inferno",
    vmin=0,
    vmax=1
)

plt.colorbar(
    label="Predicted erosion probability"
)

plt.title(
    "2015 EcoSignal Hindcast Prediction\n"
    "Predicting Observed 2015–2022 Erosion"
)

plt.axis("off")

plt.show()

In [ ]:
actual_erosion_map = np.full(
    d15_val.shape,
    np.nan
)

actual_erosion_map[
    test_eligible
] = (
    y_test_map[
        test_eligible
    ] == 1
)

plt.figure(figsize=(10, 8))

plt.imshow(
    actual_erosion_map,
    cmap="gray"
)

plt.title(
    "Actual Detectable Erosion: 2015–2022"
)

plt.axis("off")

plt.show()

In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance_df)

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    importance_df["feature"],
    importance_df["importance"]
)

plt.ylabel("Random Forest Importance")
plt.title("Hindcast Model Feature Importance")
plt.xticks(rotation=30)

plt.show()

In [ ]:
def run_threshold_summary(
    dod,
    years_between,
    rate_thresholds=[0.02, 0.03, 0.05],
    lod_values=[0.10, 0.15, 0.20]
):

    print("=== THRESHOLD SENSITIVITY ===")

    for lod in lod_values:

        for rate_threshold in rate_thresholds:

            rate = dod / years_between

            erosion = (
                (dod < -lod) &
                (rate < -rate_threshold)
            )

            print(
                f"LOD={lod:.2f}m | "
                f"rate={rate_threshold:.2f}m/yr | "
                f"erosion pixels={np.sum(erosion)}"
            )

In [ ]:
run_threshold_summary(
    dod_2015_2022,
    7
)

v2

In [ ]:
def build_hydrology_layers(dem_path):
    grid = Grid.from_raster(str(dem_path))

    dem = grid.read_raster(str(dem_path))

    # -------------------------
    # Hydrologically condition DEM
    # -------------------------

    pit_filled = grid.fill_pits(dem)

    flooded = grid.fill_depressions(
        pit_filled
    )

    inflated = grid.resolve_flats(
        flooded
    )

    # -------------------------
    # D8 flow direction
    # -------------------------

    dirmap = (
        64, 128, 1,
        2, 4, 8,
        16, 32
    )

    fdir = grid.flowdir(
        inflated,
        dirmap=dirmap
    )

    # -------------------------
    # Flow accumulation
    # -------------------------

    accumulation = grid.accumulation(
        fdir,
        dirmap=dirmap
    )

    return (
        np.asarray(inflated),
        np.asarray(fdir),
        np.asarray(accumulation)
    )

In [ ]:
!pip install pysheds -q


In [ ]:
from pysheds.grid import Grid
import numpy as np

def build_hydrology_layers(dem_path):

    # Create the PySheds grid from the GeoTIFF
    grid = Grid.from_raster(str(dem_path))

    # Read DEM into the grid
    dem = grid.read_raster(str(dem_path))

    # -------------------------
    # Hydrologically condition DEM
    # -------------------------

    pit_filled = grid.fill_pits(dem)

    depression_filled = grid.fill_depressions(
        pit_filled
    )

    conditioned_dem = grid.resolve_flats(
        depression_filled
    )

    # -------------------------
    # D8 flow directions
    # -------------------------

    dirmap = (
        64, 128, 1,
        2, 4, 8,
        16, 32
    )

    fdir = grid.flowdir(
        conditioned_dem,
        dirmap=dirmap
    )

    # -------------------------
    # Flow accumulation
    # -------------------------

    accumulation = grid.accumulation(
        fdir,
        dirmap=dirmap
    )

    return (
        np.asarray(conditioned_dem),
        np.asarray(fdir),
        np.asarray(accumulation)
    )

In [ ]:
import numpy as np
import rasterio
from pathlib import Path


# ============================================================
# TEMPORARY HYDROLOGY FILE DIRECTORY
# ============================================================

HYDRO_DIR = Path("/content/hydro_validation")
HYDRO_DIR.mkdir(parents=True, exist_ok=True)

dem13_temp = HYDRO_DIR / "dem_2013_validation.tif"
dem15_temp = HYDRO_DIR / "dem_2015_validation.tif"


# ============================================================
# FUNCTION: SAVE ALIGNED DEM AS TEMPORARY GEOTIFF
# ============================================================

def save_temp_dem(
    dem,
    transform,
    crs,
    path
):
    """
    Save an aligned DEM array as a temporary GeoTIFF
    for use with PySheds.

    Parameters
    ----------
    dem : numpy.ndarray
        DEM elevation array.

    transform : rasterio transform
        Raster transform associated with the aligned DEM.

    crs : rasterio CRS
        Coordinate reference system.

    path : Path or str
        Output GeoTIFF path.
    """

    dem = np.asarray(
        dem,
        dtype="float32"
    )

    # Replace NaN cells with explicit NoData value
    output = np.where(
        np.isfinite(dem),
        dem,
        -9999.0
    ).astype("float32")

    profile = {
        "driver": "GTiff",
        "height": output.shape[0],
        "width": output.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": crs,
        "transform": transform,
        "nodata": -9999.0,
        "compress": "deflate"
    }

    with rasterio.open(
        str(path),
        "w",
        **profile
    ) as dst:

        dst.write(
            output,
            1
        )


# ============================================================
# CREATE TEMPORARY 2013 + 2015 DEM FILES
# ============================================================

save_temp_dem(
    d13_val,
    ref_transform,
    ref_crs,
    dem13_temp
)

save_temp_dem(
    d15_val,
    ref_transform,
    ref_crs,
    dem15_temp
)


# ============================================================
# VERIFY FILES
# ============================================================

print("Temporary DEMs successfully created:")

print(
    "\n2013:",
    dem13_temp,
    "| Exists:",
    dem13_temp.exists(),
    "| Size:",
    round(
        dem13_temp.stat().st_size / 1e6,
        2
    ),
    "MB"
)

print(
    "2015:",
    dem15_temp,
    "| Exists:",
    dem15_temp.exists(),
    "| Size:",
    round(
        dem15_temp.stat().st_size / 1e6,
        2
    ),
    "MB"
)

In [ ]:
hydro_dem_2013, fdir_2013, flow_acc_2013 = build_hydrology_layers(
    dem13_temp
)

hydro_dem_2015, fdir_2015, flow_acc_2015 = build_hydrology_layers(
    dem15_temp
)

print("Hydrology layers generated.")
print(flow_acc_2013.shape)
print(flow_acc_2015.shape)

In [ ]:
# ============================================================
# RUN HYDROLOGY PROCESSING FOR 2013 AND 2015
# ============================================================

print("Processing 2013 hydrology...")

hydro_dem_2013, fdir_2013, flow_acc_2013 = build_hydrology_layers(
    dem13_temp
)

print("2013 complete.")
print("Conditioned DEM shape:", hydro_dem_2013.shape)
print("Flow direction shape:", fdir_2013.shape)
print("Flow accumulation shape:", flow_acc_2013.shape)


print("\nProcessing 2015 hydrology...")

hydro_dem_2015, fdir_2015, flow_acc_2015 = build_hydrology_layers(
    dem15_temp
)

print("2015 complete.")
print("Conditioned DEM shape:", hydro_dem_2015.shape)
print("Flow direction shape:", fdir_2015.shape)
print("Flow accumulation shape:", flow_acc_2015.shape)


# ============================================================
# BASIC QA
# ============================================================

print("\n=== HYDROLOGY QA ===")

print(
    "2013 flow accumulation range:",
    np.nanmin(flow_acc_2013),
    "to",
    np.nanmax(flow_acc_2013)
)

print(
    "2015 flow accumulation range:",
    np.nanmin(flow_acc_2015),
    "to",
    np.nanmax(flow_acc_2015)
)

print(
    "2013 matches reference shape:",
    hydro_dem_2013.shape == d13_val.shape
)

print(
    "2015 matches reference shape:",
    hydro_dem_2015.shape == d15_val.shape
)

assert hydro_dem_2013.shape == d13_val.shape
assert hydro_dem_2015.shape == d15_val.shape
assert flow_acc_2013.shape == d13_val.shape
assert flow_acc_2015.shape == d15_val.shape

print("\nPASS: Hydrology layers are aligned with both DEMs.")

In [ ]:
def build_advanced_terrain_features(
    dem,
    hydro_dem,
    flow_acc,
    transform,
    local_relief_window=101,
    drainage_percentile=97
):
    """
    Builds geomorphology + hydrology features.

    Returns:
        feature_stack
        feature_names
        diagnostic_layers
    """

    dem = np.asarray(dem, dtype="float64")
    hydro_dem = np.asarray(
        hydro_dem,
        dtype="float64"
    )

    # --------------------------------
    # Fill invalid cells
    # --------------------------------

    filled = fill_nan_nearest(dem)

    hydro_filled = np.where(
        np.isfinite(hydro_dem),
        hydro_dem,
        filled
    )

    xres = abs(transform.a)
    yres = abs(transform.e)

    cell_size = np.sqrt(
        xres * yres
    )

    # =========================================================
    # EXISTING TERRAIN FEATURES
    # =========================================================

    dz_dy, dz_dx = np.gradient(
        filled,
        yres,
        xres
    )

    slope_rad = np.arctan(
        np.sqrt(
            dz_dx**2 +
            dz_dy**2
        )
    )

    slope_deg = np.degrees(
        slope_rad
    )

    # --------------------------------
    # Local TPI
    # --------------------------------

    local_mean = uniform_filter(
        filled,
        size=101
    )

    tpi = filled - local_mean

    # --------------------------------
    # Roughness
    # --------------------------------

    rough_mean = uniform_filter(
        filled,
        size=15
    )

    rough_sq_mean = uniform_filter(
        filled**2,
        size=15
    )

    rough_var = np.maximum(
        rough_sq_mean -
        rough_mean**2,
        0
    )

    roughness = np.sqrt(
        rough_var
    )

    # =========================================================
    # NEW FEATURE 1:
    # FLOW ACCUMULATION
    # =========================================================

    flow_acc = np.asarray(
        flow_acc,
        dtype="float64"
    )

    flow_acc = np.where(
        np.isfinite(flow_acc),
        flow_acc,
        0
    )

    # log transform because accumulation
    # is extremely skewed

    log_flow_acc = np.log1p(
        flow_acc
    )

    # =========================================================
    # NEW FEATURE 2:
    # SPECIFIC CATCHMENT AREA
    # =========================================================

    # D8 accumulation = upstream cells.
    # Approximate contributing area:
    contributing_area = (
        flow_acc + 1
    ) * (
        xres * yres
    )

    # Specific catchment area:
    # area per unit contour width

    specific_catchment_area = (
        contributing_area /
        cell_size
    )

    log_sca = np.log1p(
        specific_catchment_area
    )

    # =========================================================
    # NEW FEATURE 3:
    # TOPOGRAPHIC WETNESS INDEX
    # =========================================================

    # TWI = ln(a / tan(beta))
    #
    # Avoid zero slopes

    tan_slope = np.tan(
        slope_rad
    )

    tan_slope = np.maximum(
        tan_slope,
        1e-4
    )

    twi = np.log(
        (
            specific_catchment_area +
            1e-6
        ) /
        tan_slope
    )

    # =========================================================
    # NEW FEATURES 4-5:
    # PROFILE + PLAN CURVATURE
    # =========================================================

    # First derivatives
    p = dz_dx
    q = dz_dy

    # Second derivatives
    d2z_dy2, d2z_dxdy = np.gradient(
        dz_dy,
        yres,
        xres
    )

    d2z_dydx, d2z_dx2 = np.gradient(
        dz_dx,
        yres,
        xres
    )

    r = d2z_dx2
    s = (
        d2z_dxdy +
        d2z_dydx
    ) / 2

    t = d2z_dy2

    grad_sq = (
        p**2 +
        q**2
    )

    eps = 1e-12

    # Profile curvature
    profile_curvature = -(
        r * p**2 +
        2 * s * p * q +
        t * q**2
    ) / (
        grad_sq *
        (
            1 + grad_sq
        )**1.5 +
        eps
    )

    # Plan curvature
    plan_curvature = (
        t * p**2 -
        2 * s * p * q +
        r * q**2
    ) / (
        grad_sq**1.5 +
        eps
    )

    # Limit extreme derivative artifacts
    pc_limit = np.nanpercentile(
        np.abs(
            profile_curvature[
                np.isfinite(
                    profile_curvature
                )
            ]
        ),
        99
    )

    planc_limit = np.nanpercentile(
        np.abs(
            plan_curvature[
                np.isfinite(
                    plan_curvature
                )
            ]
        ),
        99
    )

    profile_curvature = np.clip(
        profile_curvature,
        -pc_limit,
        pc_limit
    )

    plan_curvature = np.clip(
        plan_curvature,
        -planc_limit,
        planc_limit
    )

    # =========================================================
    # NEW FEATURE 6:
    # LOCAL RELIEF
    # =========================================================

    neighborhood_mean = uniform_filter(
        filled,
        size=local_relief_window
    )

    relief_residual = (
        filled -
        neighborhood_mean
    )

    local_relief = np.abs(
        relief_residual
    )

    # =========================================================
    # NEW FEATURE 7:
    # DISTANCE TO DRAINAGE
    # =========================================================

    # Define likely channels using high flow accumulation.

    positive_acc = flow_acc[
        flow_acc > 0
    ]

    if len(positive_acc) > 0:

        drainage_threshold = np.nanpercentile(
            positive_acc,
            drainage_percentile
        )

        drainage_mask = (
            flow_acc >=
            drainage_threshold
        )

    else:

        drainage_mask = np.zeros_like(
            flow_acc,
            dtype=bool
        )

    # distance_transform_edt returns
    # distance from non-drainage cells to
    # nearest drainage cell

    distance_to_drainage = (
        distance_transform_edt(
            ~drainage_mask,
            sampling=(
                yres,
                xres
            )
        )
    )

    # =========================================================
    # NEW FEATURE 8:
    # SLOPE × FLOW ACCUMULATION
    # =========================================================

    slope_flow_interaction = (
        slope_rad *
        log_flow_acc
    )

    # =========================================================
    # NORMALIZE
    # =========================================================

    elevation_norm = normalize_array(
        filled
    )

    slope_score = normalize_array(
        slope_deg
    )

    ridge_score = normalize_array(
        tpi
    )

    valley_score = normalize_array(
        -tpi
    )

    roughness_score = normalize_array(
        roughness
    )

    flow_acc_score = normalize_array(
        log_flow_acc
    )

    sca_score = normalize_array(
        log_sca
    )

    twi_score = normalize_array(
        twi
    )

    profile_curv_score = normalize_array(
        profile_curvature
    )

    plan_curv_score = normalize_array(
        plan_curvature
    )

    local_relief_score = normalize_array(
        local_relief
    )

    # Low distance = closer to drainage.
    # Keep raw direction as distance feature.
    distance_drain_score = normalize_array(
        distance_to_drainage
    )

    slope_flow_score = normalize_array(
        slope_flow_interaction
    )

    # =========================================================
    # STACK
    # =========================================================

    feature_stack = np.stack([
        elevation_norm,
        slope_score,
        ridge_score,
        valley_score,
        roughness_score,

        flow_acc_score,
        sca_score,
        twi_score,
        profile_curv_score,
        plan_curv_score,
        local_relief_score,
        distance_drain_score,
        slope_flow_score
    ], axis=-1)

    feature_names = [
        "elevation",
        "slope",
        "ridge",
        "valley",
        "roughness",

        "flow_accumulation",
        "specific_catchment_area",
        "topographic_wetness",
        "profile_curvature",
        "plan_curvature",
        "local_relief",
        "distance_to_drainage",
        "slope_x_flow"
    ]

    diagnostic_layers = {
        "flow_accumulation": flow_acc,
        "specific_catchment_area":
            specific_catchment_area,
        "twi": twi,
        "profile_curvature":
            profile_curvature,
        "plan_curvature":
            plan_curvature,
        "local_relief":
            local_relief,
        "drainage_mask":
            drainage_mask,
        "distance_to_drainage":
            distance_to_drainage,
        "slope_x_flow":
            slope_flow_interaction
    }

    return (
        feature_stack,
        feature_names,
        diagnostic_layers
    )

In [ ]:
X_2013_advanced, advanced_feature_names, diag_2013 = (
    build_advanced_terrain_features(
        d13_val,
        hydro_dem_2013,
        flow_acc_2013,
        ref_transform
    )
)

X_2015_advanced, _, diag_2015 = (
    build_advanced_terrain_features(
        d15_val,
        hydro_dem_2015,
        flow_acc_2015,
        ref_transform
    )
)

print(
    "Advanced feature shape:",
    X_2013_advanced.shape
)

print(
    "Features:"
)

for feature in advanced_feature_names:
    print("-", feature)

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(
    np.log1p(
        diag_2015[
            "flow_accumulation"
        ]
    ),
    cmap="viridis"
)

plt.colorbar(
    label="Log flow accumulation"
)

plt.title(
    "2015 Flow Accumulation"
)

plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(
    diag_2015["twi"],
    cmap="Blues"
)

plt.colorbar(
    label="Topographic Wetness Index"
)

plt.title(
    "2015 Topographic Wetness Index"
)

plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(
    diag_2015[
        "distance_to_drainage"
    ],
    cmap="magma"
)

plt.colorbar(
    label="Distance to drainage (m)"
)

plt.title(
    "2015 Distance to Derived Drainage"
)

plt.axis("off")
plt.show()

In [ ]:
y_train_map
train_eligible

y_test_map
test_eligible

In [ ]:
feature_valid_2013_adv = np.all(
    np.isfinite(
        X_2013_advanced
    ),
    axis=-1
)

valid_train_adv = (
    train_eligible &
    feature_valid_2013_adv &
    np.isfinite(d13_val) &
    np.isfinite(d15_val)
)

X_train_adv_all = (
    X_2013_advanced[
        valid_train_adv
    ]
)

y_train_adv_all = (
    y_train_map[
        valid_train_adv
    ]
)

print(
    "Available advanced training pixels:",
    len(y_train_adv_all)
)

In [ ]:
rng = np.random.default_rng(42)

MAX_TRAIN_PIXELS = 300_000

if len(
    y_train_adv_all
) > MAX_TRAIN_PIXELS:

    selected = rng.choice(
        len(
            y_train_adv_all
        ),
        size=
            MAX_TRAIN_PIXELS,
        replace=False
    )

    X_train_adv = (
        X_train_adv_all[
            selected
        ]
    )

    y_train_adv = (
        y_train_adv_all[
            selected
        ]
    )

else:

    X_train_adv = (
        X_train_adv_all
    )

    y_train_adv = (
        y_train_adv_all
    )

print(
    "Advanced training pixels:",
    len(y_train_adv)
)

print(
    "Erosion prevalence:",
    np.mean(
        y_train_adv
    )
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_advanced = RandomForestClassifier(
    n_estimators=500,
    max_depth=22,
    min_samples_leaf=8,

    max_features="sqrt",

    class_weight=
        "balanced_subsample",

    n_jobs=-1,
    random_state=42,

    oob_score=True
)

rf_advanced.fit(
    X_train_adv,
    y_train_adv
)

print(
    "Advanced OOB accuracy:",
    rf_advanced.oob_score_
)

In [ ]:
rf_advanced = RandomForestClassifier(
    n_estimators=500,
    max_depth=22,
    min_samples_leaf=8,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
    oob_score=True
)

rf_advanced.fit(
    X_train_adv,
    y_train_adv
)

print("Advanced OOB accuracy:", rf_advanced.oob_score_)

In [ ]:
oob_prob_adv = rf_advanced.oob_decision_function_[:, 1]
valid_oob_adv = np.isfinite(oob_prob_adv)

precision_adv, recall_adv, thresholds_adv = precision_recall_curve(
    y_train_adv[valid_oob_adv],
    oob_prob_adv[valid_oob_adv]
)

f1_adv = (
    2 * precision_adv * recall_adv / (precision_adv + recall_adv + 1e-12)
)

best_adv_index = np.nanargmax(f1_adv[:-1])
decision_threshold_adv = thresholds_adv[best_adv_index]

print("Advanced decision threshold:", decision_threshold_adv)
print("Advanced training OOB F1:", f1_adv[best_adv_index])

In [ ]:
from sklearn.metrics import (
    precision_recall_curve
)

oob_prob_adv = (
    rf_advanced
    .oob_decision_function_[:, 1]
)

valid_oob_adv = np.isfinite(
    oob_prob_adv
)

precision_adv, recall_adv, thresholds_adv = (
    precision_recall_curve(
        y_train_adv[
            valid_oob_adv
        ],
        oob_prob_adv[
            valid_oob_adv
        ]
    )
)

f1_adv = (
    2 *
    precision_adv *
    recall_adv /
    (
        precision_adv +
        recall_adv +
        1e-12
    )
)

best_adv_index = np.nanargmax(
    f1_adv[:-1]
)

decision_threshold_adv = (
    thresholds_adv[
        best_adv_index
    ]
)

print(
    "Advanced decision threshold:",
    decision_threshold_adv
)

print(
    "Advanced training OOB F1:",
    f1_adv[
        best_adv_index
    ]
)

In [ ]:
feature_valid_2015_adv = np.all(
    np.isfinite(
        X_2015_advanced
    ),
    axis=-1
)

valid_test_adv = (
    test_eligible &
    feature_valid_2015_adv &
    np.isfinite(d15_val) &
    np.isfinite(d22_val)
)

X_test_adv = (
    X_2015_advanced[
        valid_test_adv
    ]
)

y_test_adv = (
    y_test_map[
        valid_test_adv
    ]
)

test_probability_adv = (
    predict_probability_batches(
        rf_advanced,
        X_test_adv
    )
)

test_prediction_adv = (
    test_probability_adv >=
    decision_threshold_adv
).astype("int8")

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

roc_auc_adv = roc_auc_score(
    y_test_adv,
    test_probability_adv
)

pr_auc_adv = average_precision_score(
    y_test_adv,
    test_probability_adv
)

precision_value_adv = precision_score(
    y_test_adv,
    test_prediction_adv,
    zero_division=0
)

recall_value_adv = recall_score(
    y_test_adv,
    test_prediction_adv,
    zero_division=0
)

f1_value_adv = f1_score(
    y_test_adv,
    test_prediction_adv,
    zero_division=0
)

cm_adv = confusion_matrix(
    y_test_adv,
    test_prediction_adv
)

print(
    "=== ADVANCED TEMPORAL HINDCAST ==="
)

print(
    "ROC-AUC:",
    round(
        roc_auc_adv,
        4
    )
)

print(
    "PR-AUC:",
    round(
        pr_auc_adv,
        4
    )
)

print(
    "Precision:",
    round(
        precision_value_adv,
        4
    )
)

print(
    "Recall:",
    round(
        recall_value_adv,
        4
    )
)

print(
    "F1:",
    round(
        f1_value_adv,
        4
    )
)

print(
    "\nConfusion matrix:"
)

print(cm_adv)

top_threshold_adv = np.percentile(
    test_probability_adv,
    90
)

top_risk_adv = (
    test_probability_adv >=
    top_threshold_adv
)

top_precision_adv = np.mean(
    y_test_adv[
        top_risk_adv
    ]
)

baseline_adv = np.mean(
    y_test_adv
)

lift_adv = (
    top_precision_adv /
    baseline_adv
)

capture_adv = (
    np.sum(
        (y_test_adv == 1) &
        top_risk_adv
    ) /
    np.sum(
        y_test_adv == 1
    )
)

print(
    "=== ADVANCED TOP 10% TEST ==="
)

print(
    "Baseline erosion rate:",
    round(
        baseline_adv,
        4
    )
)

print(
    "Erosion rate inside top 10%:",
    round(
        top_precision_adv,
        4
    )
)

print(
    "Risk enrichment/lift:",
    round(
        lift_adv,
        2
    ),
    "x"
)

print(
    "% observed erosion captured:",
    round(
        capture_adv * 100,
        2
    ),
    "%"
)

In [ ]:
importance_adv = pd.DataFrame({
    "feature":
        advanced_feature_names,

    "importance":
        rf_advanced
        .feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance_adv

In [ ]:
plt.figure(
    figsize=(10, 6)
)

plt.bar(
    importance_adv[
        "feature"
    ],
    importance_adv[
        "importance"
    ]
)

plt.ylabel(
    "Random Forest importance"
)

plt.title(
    "Advanced Hindcast Feature Importance"
)

plt.xticks(
    rotation=60,
    ha="right"
)

plt.tight_layout()
plt.show()

v3

In [ ]:
# ============================================================
# FINAL CONTROLLED MODEL TEST:
# HISTOGRAM GRADIENT BOOSTING VS ADVANCED RANDOM FOREST
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve
)

print("============================================================")
print("FINAL CONTROLLED GRADIENT-BOOSTING TEST")
print("============================================================")

# ------------------------------------------------------------
# 1. CHECK EXISTING DATA
# ------------------------------------------------------------

print("\nTraining shape:", X_train_adv.shape)
print("Testing shape:", X_test_adv.shape)
print("Number of features:", X_train_adv.shape[1])
print("Training erosion prevalence:", round(np.mean(y_train_adv), 4))
print("Testing erosion prevalence:", round(np.mean(y_test_adv), 4))


# ------------------------------------------------------------
# 2. SPLIT ONLY THE 2013–2015 TRAINING DATA
#
# This internal validation split is used ONLY to choose the
# classification threshold.
#
# 2015–2022 remains untouched.
# ------------------------------------------------------------

X_gb_train, X_gb_val, y_gb_train, y_gb_val = train_test_split(
    X_train_adv,
    y_train_adv,
    test_size=0.20,
    stratify=y_train_adv,
    random_state=42
)

print("\nInternal training pixels:", len(y_gb_train))
print("Internal validation pixels:", len(y_gb_val))


# ------------------------------------------------------------
# 3. HANDLE CLASS IMBALANCE WITH SAMPLE WEIGHTS
# ------------------------------------------------------------

n_negative = np.sum(y_gb_train == 0)
n_positive = np.sum(y_gb_train == 1)

positive_weight = n_negative / max(n_positive, 1)

sample_weights = np.where(
    y_gb_train == 1,
    positive_weight,
    1.0
)

print(
    "Positive-class training weight:",
    round(positive_weight, 2)
)


# ------------------------------------------------------------
# 4. TRAIN HISTOGRAM GRADIENT BOOSTING
# ------------------------------------------------------------

gb_model = HistGradientBoostingClassifier(
    learning_rate=0.06,
    max_iter=350,
    max_leaf_nodes=31,
    max_depth=None,
    min_samples_leaf=30,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=25,
    random_state=42
)

print("\nTraining gradient-boosted model...")

start_time = time.time()

gb_model.fit(
    X_gb_train,
    y_gb_train,
    sample_weight=sample_weights
)

elapsed = time.time() - start_time

print(
    f"Training complete in {elapsed / 60:.2f} minutes."
)

print(
    "Boosting iterations used:",
    gb_model.n_iter_
)


# ------------------------------------------------------------
# 5. SELECT DECISION THRESHOLD USING ONLY TRAINING-PERIOD DATA
# ------------------------------------------------------------

gb_val_probability = gb_model.predict_proba(
    X_gb_val
)[:, 1]

precision_curve, recall_curve, thresholds_curve = (
    precision_recall_curve(
        y_gb_val,
        gb_val_probability
    )
)

f1_curve = (
    2 *
    precision_curve *
    recall_curve /
    (
        precision_curve +
        recall_curve +
        1e-12
    )
)

best_threshold_index = np.nanargmax(
    f1_curve[:-1]
)

gb_threshold = thresholds_curve[
    best_threshold_index
]

print(
    "\nTraining-period selected threshold:",
    round(float(gb_threshold), 4)
)

print(
    "Internal validation F1 at threshold:",
    round(
        float(f1_curve[best_threshold_index]),
        4
    )
)


# ------------------------------------------------------------
# 6. TEST ON UNTOUCHED 2015–2022 DATA
# ------------------------------------------------------------

print("\nRunning untouched 2015–2022 hindcast test...")

gb_test_probability = gb_model.predict_proba(
    X_test_adv
)[:, 1]

gb_test_prediction = (
    gb_test_probability >= gb_threshold
).astype("int8")


# ------------------------------------------------------------
# 7. STANDARD METRICS
# ------------------------------------------------------------

gb_roc_auc = roc_auc_score(
    y_test_adv,
    gb_test_probability
)

gb_pr_auc = average_precision_score(
    y_test_adv,
    gb_test_probability
)

gb_precision = precision_score(
    y_test_adv,
    gb_test_prediction,
    zero_division=0
)

gb_recall = recall_score(
    y_test_adv,
    gb_test_prediction,
    zero_division=0
)

gb_f1 = f1_score(
    y_test_adv,
    gb_test_prediction,
    zero_division=0
)

gb_cm = confusion_matrix(
    y_test_adv,
    gb_test_prediction
)


# ------------------------------------------------------------
# 8. TOP-10% RISK TEST
# ------------------------------------------------------------

gb_top_threshold = np.percentile(
    gb_test_probability,
    90
)

gb_top_risk = (
    gb_test_probability >= gb_top_threshold
)

gb_baseline = np.mean(
    y_test_adv
)

gb_top_erosion_rate = np.mean(
    y_test_adv[
        gb_top_risk
    ]
)

gb_lift = (
    gb_top_erosion_rate /
    gb_baseline
)

gb_capture = (
    np.sum(
        (y_test_adv == 1) &
        gb_top_risk
    ) /
    np.sum(
        y_test_adv == 1
    )
)


# ------------------------------------------------------------
# 9. PRINT RESULTS
# ------------------------------------------------------------

print("\n============================================================")
print("GRADIENT-BOOSTED TEMPORAL HINDCAST")
print("============================================================")

print(
    "ROC-AUC:",
    round(gb_roc_auc, 4)
)

print(
    "PR-AUC:",
    round(gb_pr_auc, 4)
)

print(
    "Precision:",
    round(gb_precision, 4)
)

print(
    "Recall:",
    round(gb_recall, 4)
)

print(
    "F1:",
    round(gb_f1, 4)
)

print("\nConfusion matrix:")
print(gb_cm)


print("\n============================================================")
print("GRADIENT-BOOSTED TOP 10% TEST")
print("============================================================")

print(
    "Baseline erosion rate:",
    round(gb_baseline, 4)
)

print(
    "Erosion rate inside top 10%:",
    round(gb_top_erosion_rate, 4)
)

print(
    "Risk enrichment/lift:",
    round(gb_lift, 2),
    "x"
)

print(
    "% observed erosion captured:",
    round(gb_capture * 100, 2),
    "%"
)


# ------------------------------------------------------------
# 10. COMPARE WITH ADVANCED RANDOM FOREST
# ------------------------------------------------------------

rf_results = {
    "ROC-AUC": 0.7624,
    "PR-AUC": 0.2069,
    "Precision": 0.3125,
    "Recall": 0.2064,
    "F1": 0.2486,
    "Top-10% Lift": 3.71,
    "Top-10% Capture (%)": 37.08
}

gb_results = {
    "ROC-AUC": gb_roc_auc,
    "PR-AUC": gb_pr_auc,
    "Precision": gb_precision,
    "Recall": gb_recall,
    "F1": gb_f1,
    "Top-10% Lift": gb_lift,
    "Top-10% Capture (%)": gb_capture * 100
}

comparison = pd.DataFrame({
    "Advanced Random Forest": rf_results,
    "Gradient Boosting": gb_results
})

comparison["GB - RF"] = (
    comparison["Gradient Boosting"] -
    comparison["Advanced Random Forest"]
)

print("\n============================================================")
print("FINAL MODEL COMPARISON")
print("============================================================")

display(
    comparison.round(4)
)


# ------------------------------------------------------------
# 11. AUTOMATIC RECOMMENDATION
# ------------------------------------------------------------

rf_primary_score = (
    0.30 * rf_results["ROC-AUC"] +
    0.30 * rf_results["PR-AUC"] +
    0.20 * (rf_results["Top-10% Lift"] / 4.0) +
    0.20 * (rf_results["Top-10% Capture (%)"] / 40.0)
)

gb_primary_score = (
    0.30 * gb_results["ROC-AUC"] +
    0.30 * gb_results["PR-AUC"] +
    0.20 * (gb_results["Top-10% Lift"] / 4.0) +
    0.20 * (gb_results["Top-10% Capture (%)"] / 40.0)
)

print("\n============================================================")
print("FINAL DECISION")
print("============================================================")

if gb_primary_score > rf_primary_score * 1.02:

    print(
        "KEEP GRADIENT BOOSTING."
    )

    print(
        "It materially improves the primary risk-ranking metrics."
    )

elif rf_primary_score > gb_primary_score * 1.02:

    print(
        "KEEP ADVANCED RANDOM FOREST."
    )

    print(
        "Gradient boosting did not materially improve the validated model."
    )

else:

    print(
        "PERFORMANCE IS VERY SIMILAR."
    )

    print(
        "Keep the Advanced Random Forest because it is simpler "
        "and already validated."
    )

print("\nRandom Forest score:", round(rf_primary_score, 4))
print("Gradient Boosting score:", round(gb_primary_score, 4))

In [ ]:
import joblib
import json
import sklearn
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")

# Verify required variables
required = [
    "rf_advanced",
    "advanced_feature_names",
    "decision_threshold_adv",
    "X_train_adv",
    "roc_auc_adv",
    "pr_auc_adv"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing variables: " + ", ".join(missing) +
        "\nRun the necessary earlier cells first."
    )

assert hasattr(rf_advanced, "estimators_"), \
    "Random Forest is not fitted."

assert rf_advanced.n_features_in_ == len(
    advanced_feature_names
), "Feature count mismatch."

# Save to Google Drive
SAVE_DIR = Path(
    "/content/drive/MyDrive/EcoSignal_DEM/"
    "geomorphology_v1_backup"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

model_path = SAVE_DIR / "geomorphology_rf_v1.joblib"
metadata_path = SAVE_DIR / "model_metadata.json"

# 1. Save trained model
joblib.dump(rf_advanced, model_path, compress=3)

# 2. Save model configuration and results
metadata = {
    "model_name": "EcoSignal Geomorphology V1",
    "model_type": "RandomForestClassifier",
    "features": list(advanced_feature_names),
    "n_features": len(advanced_feature_names),
    "decision_threshold": float(
        decision_threshold_adv
    ),
    "parameters": rf_advanced.get_params(),
    "training_period": "2013-2015",
    "validation_period": "2015-2022",
    "roc_auc": float(roc_auc_adv),
    "pr_auc": float(pr_auc_adv),
    "sklearn_version": sklearn.__version__,
    "saved_at_utc": datetime.now(
        timezone.utc
    ).isoformat()}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)

# 3. Store sample predictions to check reloading
rng = np.random.default_rng(2026)
sample_idx = rng.choice(
    len(X_train_adv),
    size=min(100, len(X_train_adv)),
    replace=False
)

sample_X = X_train_adv[sample_idx]
sample_prob = rf_advanced.predict_proba(sample_X)

np.savez_compressed(
    SAVE_DIR / "model_verification.npz",
    X=sample_X,
    probabilities=sample_prob
)

print("\n" + "=" * 55)
print("ECOSIGNAL V1 MODEL SAVED SUCCESSFULLY")
print("=" * 55)
print("Location:", SAVE_DIR)
print("Features:", len(advanced_feature_names))
print("Trees:", rf_advanced.n_estimators)
print("ROC-AUC:", round(float(roc_auc_adv), 4))
print("PR-AUC:", round(float(pr_auc_adv), 4))
print("Model size:",
      round(model_path.stat().st_size / 1e6, 2),
      "MB")

In [ ]:
import joblib
import json
import sklearn
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")

# Verify required variables
required = [
    "rf_advanced",
    "advanced_feature_names",
    "decision_threshold_adv",
    "X_train_adv",
    "roc_auc_adv",
    "pr_auc_adv"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing variables: " + ", ".join(missing) +
        "\nRun the necessary earlier cells first."
    )

assert hasattr(rf_advanced, "estimators_"), \
    "Random Forest is not fitted."

assert rf_advanced.n_features_in_ == len(
    advanced_feature_names
), "Feature count mismatch."

# Save to Google Drive
SAVE_DIR = Path(
    "/content/drive/MyDrive/EcoSignal_DEM/"
    "geomorphology_v1_backup"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

model_path = SAVE_DIR / "geomorphology_rf_v1.joblib"
metadata_path = SAVE_DIR / "model_metadata.json"

# 1. Save trained model
joblib.dump(rf_advanced, model_path, compress=3)

# 2. Save model configuration and results
metadata = {
    "model_name": "EcoSignal Geomorphology V1",
    "model_type": "RandomForestClassifier",
    "features": list(advanced_feature_names),
    "n_features": len(advanced_feature_names),
    "decision_threshold": float(
        decision_threshold_adv
    ),
    "parameters": rf_advanced.get_params(),
    "training_period": "2013-2015",
    "validation_period": "2015-2022",
    "roc_auc": float(roc_auc_adv),
    "pr_auc": float(pr_auc_adv),
    "sklearn_version": sklearn.__version__,
    "saved_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2, default=str)

# 3. Store sample predictions to check reloading
rng = np.random.default_rng(2026)
sample_idx = rng.choice(
    len(X_train_adv),
    size=min(100, len(X_train_adv)),
    replace=False
)

sample_X = X_train_adv[sample_idx]
sample_prob = rf_advanced.predict_proba(sample_X)

np.savez_compressed(
    SAVE_DIR / "model_verification.npz",
    X=sample_X,
    probabilities=sample_prob
)

print("\n" + "=" * 55)
print("ECOSIGNAL V1 MODEL SAVED SUCCESSFULLY")
print("=" * 55)
print("Location:", SAVE_DIR)
print("Features:", len(advanced_feature_names))
print("Trees:", rf_advanced.n_estimators)
print("ROC-AUC:", round(float(roc_auc_adv), 4))
print("PR-AUC:", round(float(pr_auc_adv), 4))
print("Model size:",
      round(model_path.stat().st_size / 1e6, 2),
      "MB")

In [ ]:
import joblib
import numpy as np
import json

loaded_model = joblib.load(model_path)

with open(metadata_path) as f:
    loaded_metadata = json.load(f)

with np.load(
    SAVE_DIR / "model_verification.npz"
) as data:
    sample_X = data["X"]
    expected_prob = data["probabilities"]

actual_prob = loaded_model.predict_proba(sample_X)

np.testing.assert_allclose(
    expected_prob,
    actual_prob,
    rtol=1e-10,
    atol=1e-12
)

assert loaded_model.n_features_in_ == \
    loaded_metadata["n_features"]

print("MODEL VERIFICATION PASSED")
print("Predictions match exactly within tolerance.")
print("Saved features:",
      loaded_model.n_features_in_)
print("Saved trees:",
      loaded_model.n_estimators)
print("Backup:", SAVE_DIR)


Debug

In [ ]:
import requests

item_url = "https://www.sciencebase.gov/catalog/item/63501ed5d34e47431c15c189?format=json"

item = requests.get(item_url).json()

for f in item.get("files", []):
    if f.get("name") == "Dem_2013.tif":
        print("DIRECT TIF URL:")
        print(f.get("url"))

In [ ]:
import requests

url = "https://sciencebase.usgs.gov/manager/item/63501ed5d34e47431c15c189/file/clcjahqpz000v0rnz7qf1gvbo"
output_path = "/content/Dem_2013.tif"

with requests.get(url, stream=True) as r:
    r.raise_for_status()

    total = int(r.headers.get("content-length", 0))
    downloaded = 0

    with open(output_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024 * 8):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

                if total:
                    print(
                        f"\rDownloaded "
                        f"{downloaded / 1e9:.2f} / {total / 1e9:.2f} GB "
                        f"({downloaded / total * 100:.1f}%) -- {downloaded} bytes",
                        end=""
                    )
                else:
                    print(f"\rDownloaded {downloaded} bytes", end="")

print("\nSaved to:", output_path)